# Observing Phenometrics in Madagascar using MODIS data
---
**Summary**
This script demonstrates how to access and use Moderate Resolution Imaging Spectroradiometer (MODIS) NDVI, EVI, QA, and DOY products through Earthdata. Specifically, the code searches for products in the bounds of the Military Grid Reference System (MGRS) tile, 39KUA, in order to compare its results with an HLS script that examines the same region.

**What is MODIS data?**
The Moderate Resolution Imaging Spectroradiometer (MODIS) continually collects data in 36 spectral channels with global coverage every 1 to 2 days. Its exceptionally broad spectral range enables MODIS data to be used in studies across numerous disciplines, including vegetative health, changes in land cover and land use, oceans and ocean biology, sea surface temperature, and cloud analysis. It also is used extensively for monitoring fires and natural hazards along with oil spills. 

An important attribute of MODIS data is the availability of MODIS data products in real-time and near real-time. Direct broadcast stations around the world download raw MODIS data in real-time directly from the satellite, while NASA’s Land, Atmosphere Near Real-time Capability for EOS (LANCE) provides several MODIS products within three hours of satellite observation.

This notebook uses MODIS composite images of 16-day periods from 2016-2025.

(https://www.earthdata.nasa.gov/data/instruments/modis)

Phenometrics: phenological (cyclic and natural phenomena) transition dates

## Tutorial Outline

1. Environment Setup
2. Initialize variables and grab HDF files from Earthdata
3. Clean data and convert HDF to TIFF
4. MODIS and HLS comparison

## 1. Environment Setup
**Environment:** TBA

This section sets the libraries to access the MODIS data through Earthdata as well as the libraries for the creating the bounding box and processing the data. Note: to use earthaccess to interface with Earthdata, you will need an Earthdata account (https://urs.earthdata.nasa.gov).

In [1]:
import subprocess, sys

PACKAGES = [
    "earthaccess", "rioxarray", "rasterio", "xarray",
    "numpy", "pandas", "matplotlib", "pyproj",
    "shapely", "geopandas", "tqdm", "mgrs"
]

for pkg in PACKAGES:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f"✓ Installed {pkg}")
    except:
        print(f"⚠ Could not install {pkg} (may already exist or require fallback)")

print("✅ All packages installed.")

✓ Installed earthaccess
✓ Installed rioxarray
✓ Installed rasterio
✓ Installed xarray
✓ Installed numpy
✓ Installed pandas
✓ Installed matplotlib
✓ Installed pyproj
✓ Installed shapely
✓ Installed geopandas
✓ Installed tqdm
✓ Installed mgrs
✅ All packages installed.


In [3]:
import os, stat, getpass, earthaccess
from pathlib import Path

# Authenticate with NASA Earthdata
def authenticate_earthdata():
    """
    Robust three-strategy authentication chain for NASA Earthdata
    Strategies:
        1. Environment variables (EARTHDATA_USERNAME, EARTHDATA_PASSWORD)
        2. .netrc file in home directory
        3. Interactive prompt
    """
    print("Authenticating with NASA Earthdata...")

    # Strategy 1: Environment variables
    if 'EARTHDATA_USERNAME' in os.environ and 'EARTHDATA_PASSWORD' in os.environ:
        print("  → Using environment variables")
        try:
            auth = earthaccess.login(strategy="environment")
            if auth.authenticated:
                print("  ✓ Authentication successful (environment)")
                return auth
        except Exception as e:
            print(f"  ⚠ Environment authentication failed: {e}")
    
    # Strategy 2: .netrc file
    netrc_path = Path.home() / '.netrc'
    if netrc_path.exists():
        print("  → Using .netrc file")
        try:
            auth = earthaccess.login(strategy="netrc")
            if auth.authenticated:
                print("  ✓ Authentication successful (.netrc)")
                return auth
        except Exception as e:
            print(f"  ⚠ .netrc authentication failed: {e}")
    
    # Strategy 3: Interactive prompt
    print("  → Using interactive prompt")
    try:
        auth = earthaccess.login(strategy="interactive")
        if auth.authenticated:
            print("  ✓ Authentication successful (interactive)")
            return auth
    except Exception as e:
        print(f"  ✗ Interactive authentication failed: {e}")
        raise
    
    raise RuntimeError("All authentication strategies failed")

auth = authenticate_earthdata()

Authenticating with NASA Earthdata...
  → Using interactive prompt


Enter your Earthdata Login username:  tmtle
Enter your Earthdata password:  ········


  ✓ Authentication successful (interactive)


## 2. Initialize variables and grab HDF files from Earthdata
This section sets up the configuration, such as the MGRS_TILE, START_DATE, END_DATE, and more. A shp file is also required if there is a specific region of interest that should define the bounding box.

After the configuration is set up, the code will search Earthdata for HDF files that meet the configuration requirements and download.

In [4]:
import re, warnings
import numpy  as np
import pandas as pd
from pathlib           import Path
from shapely.geometry  import box, mapping, shape, Polygon
from shapely.ops       import unary_union
from pyproj            import Transformer
import geopandas as gpd

# ─────────────────────────────────────────────────────────────────────────────
# USER INPUTS
# ─────────────────────────────────────────────────────────────────────────────

MGRS_TILE       = "39KUA"        # ← Corrected MGRS tile for Betampona
START_DATE      = "2016-01-01"
END_DATE        = "2025-12-31"
PRODUCT_CHOICE  = "NDVI"         # "NDVI" | "NBAR" | "BOTH"
MODIS_PLATFORM  = "Terra"        # "Terra" | "Aqua" | "Both"
NDVI_RESOLUTION = "250m"         # "250m" → MOD13Q1 | "1km" → MOD13A2
TIFF_FORMAT     = "COG"          # "COG" | "STANDARD"

# ── Output directories ────────────────────────────────────────────────────────
DIR_OUT = f"modis_veg_indices/{MGRS_TILE}"
DIR_HDF = Path(DIR_OUT) / "modis_hdf"
# DIR_HDF  = f"./modis_hdf/{MGRS_TILE}"
# DIR_TIFF = f"./modis_tiff/{MGRS_TILE}"
DIR_TIFF = Path(DIR_OUT) / "modis_tiff"
FIG_DIR = Path(DIR_OUT) / "modis_fig"

# ─────────────────────────────────────────────────────────────────────────────
# SITE: RNI Betampona
# Source    : Betampona.shp (EPSG:32739 → reprojected to WGS84)
# Features  : 6 polygons dissolved to single boundary
# Area      : ~2,223 ha primary rainforest or ~44100 ha
# Location  : ~50km NW of Toamasina, eastern Madagascar
# ─────────────────────────────────────────────────────────────────────────────
# SITE: Carter Country
# Source    : ccm_roi.shp (EPSG:4326 → reprojected to WGS84)
# Features  : 2 polygons dissolved to single boundary
# Area      : ~16187 ha primary shrub steppe
# ─────────────────────────────────────────────────────────────────────────────
# SITE: 18SUJ Tile
# Source    : patuxent.zip
# Features  : 1 polygons dissolved to single boundary
# Area      : ~49450 ha
# ─────────────────────────────────────────────────────────────────────────────
SITE_NAME    = "Betampona"
SITE_AREA_HA = 44100

# ── Load shapefile and dissolve to single ROI ─────────────────────────────────
SHP_PATH = "/shared/users-local/tl3uk/tommy_code/roi_shp/betampona.zip"
#"/shared/users/hls_bdec/tommy_code/roi_shp/patuxent.zip"
#"/shared/users/hls_bdec/amanda_code/ccm_roi/ccm_roi.shp"
#"/shared/users/hls_bdec/amanda_code/betampona_roi/betampona06.shp"   # extracted in diagnostic cell

try:
    if Path(SHP_PATH).suffix == ".zip":
        gdf_raw = gpd.read_file(f"zip://{SHP_PATH}")
    else:
        gdf_raw = gpd.read_file(SHP_PATH)

    # Reproject to WGS84
    source_crs = gdf_raw.crs.to_epsg()
    if source_crs != 4326:
        gdf_raw = gdf_raw.to_crs("EPSG:4326")

    # Dissolve all 6 features to one boundary
    gdf_dissolved  = gdf_raw.dissolve()
    ROI  = gdf_dissolved.geometry.iloc[0]
    print(f"✅ Shapefile loaded: {len(gdf_raw)} features dissolved to 1 polygon")
    print(f"   Source CRS : EPSG:{source_crs}")
    print(f"   Working CRS: EPSG:4326 (WGS84)")

except Exception as e:
    print(f"⚠️  Could not load shapefile ({e})")
    print(f"   Falling back to MGRS tile bounds")
    # Fallback: use exact bounds from diagnostic output
    # from shapely.geometry import box as shp_box
    # ROI = shp_box(49.195236, -17.932427, 49.248102, -17.873555)
    ROI = None

if not ROI is None:
    min_lon, min_lat, max_lon, max_lat = ROI.bounds

    # ── Exact bounds from shapefile ───────────────────────────────────────────────
    ROI_BBOX = (
        min_lon, # 49.195236,    # min_lon (W)
        min_lat, # -17.932427,    # min_lat (S)
        max_lon, # 49.248102,    # max_lon (E)
        max_lat, # -17.873555,    # max_lat (N)
    )
    
    # Might be a bit off, since it's not in a projected CRS
    ROI_CENTROID = gdf_dissolved.geometry.centroid[0] # (49.215539, -17.903742)   # (lon, lat)
    ROI_CENTROID = (ROI_CENTROID.x, ROI_CENTROID.y)
    
    # Use Betampona as the CMR search ROI
    SEARCH_BBOX = ROI_BBOX

# ─────────────────────────────────────────────────────────────────────────────
# PRODUCTS TO FETCH
# ─────────────────────────────────────────────────────────────────────────────
_PRODUCT_MAP = {
    "NDVI": {
        "Terra": {"250m": [("MOD13Q1","061")], "1km": [("MOD13A2","061")]},
        "Aqua" : {"250m": [("MYD13Q1","061")], "1km": [("MYD13A2","061")]},
        "Both" : {"250m": [("MOD13Q1","061"),("MYD13Q1","061")],
                  "1km" : [("MOD13A2","061"),("MYD13A2","061")]},
    },
    "NBAR": {
        "Terra": [("MOD09GA","006")],
        "Aqua" : [("MYD09GA","006")],
        "Both" : [("MOD09GA","006"),("MYD09GA","006")],
    },
}

PRODUCTS_TO_FETCH = []
if PRODUCT_CHOICE in ("NDVI","BOTH"):
    PRODUCTS_TO_FETCH += _PRODUCT_MAP["NDVI"][MODIS_PLATFORM][NDVI_RESOLUTION]
if PRODUCT_CHOICE in ("NBAR","BOTH"):
    PRODUCTS_TO_FETCH += _PRODUCT_MAP["NBAR"][MODIS_PLATFORM]

# ─────────────────────────────────────────────────────────────────────────────
# MGRS TILE → WGS84 BBOX
# ─────────────────────────────────────────────────────────────────────────────
MGRS_LAT_BANDS = {
    "C":-80,"D":-72,"E":-64,"F":-56,"G":-48,"H":-40,
    "J":-32,"K":-24,"L":-16,"M": -8,"N":  0,"P":  8,
    "Q": 16,"R": 24,"S": 32,"T": 40,"U": 48,"V": 56,
    "W": 64,"X": 72,
}

def _mgrs_bbox_via_package(tile, pad=0.05):
    import mgrs as _mgrs
    m = _mgrs.MGRS()
    lons, lats = [], []
    for e in [0,1,50000,99998,99999]:
        for n in [0,1,50000,99998,99999]:
            try:
                lat, lon = m.toLatLon(
                    f"{tile}{str(e).zfill(5)}{str(n).zfill(5)}"
                )
                lons.append(lon); lats.append(lat)
            except Exception:
                continue
    if not lons:
        raise ValueError(f"mgrs could not parse: {tile}")
    return (round(min(lons)-pad,6), round(min(lats)-pad,6),
            round(max(lons)+pad,6), round(max(lats)+pad,6))

def _mgrs_bbox_via_pyproj(tile, pad=0.05):
    tile = tile.strip().upper()
    m = re.match(
        r'^(\d{1,2})([C-HJ-NP-X])([A-HJ-NP-Z])([A-HJ-NP-V])$', tile
    )
    if not m:
        raise ValueError(f"Cannot parse MGRS tile: {tile}")
    zone_num = int(m.group(1)); lat_band = m.group(2)
    col_let  = m.group(3);     row_let  = m.group(4)
    hemi_    = "N" if MGRS_LAT_BANDS.get(lat_band,0) >= 0 else "S"
    epsg_    = (32600 if hemi_=="N" else 32700) + zone_num
    col_set  = "ABCDEFGH" if zone_num%2==1 else "JKLMNPQR"
    row_set  = "ABCDEFGHJKLMNPQRSTUV"
    if col_let not in col_set:
        raise ValueError(
            f"Column letter '{col_let}' invalid for zone {zone_num}"
        )
    e_sw = (col_set.index(col_let)+1)*100_000
    n_sw = row_set.index(row_let)*100_000
    if MGRS_LAT_BANDS.get(lat_band,0) < 0:
        n_sw += int((MGRS_LAT_BANDS[lat_band]+80)/8)*2_000_000
    tf = Transformer.from_crs(f"EPSG:{epsg_}","EPSG:4326",always_xy=True)
    lons, lats = [], []
    for e,n in [(e_sw,n_sw),(e_sw+100000,n_sw),(e_sw,n_sw+100000),
                (e_sw+100000,n_sw+100000),(e_sw+50000,n_sw+50000)]:
        try:
            lon,lat = tf.transform(e,n)
            if -180<=lon<=180 and -90<=lat<=90:
                lons.append(lon); lats.append(lat)
        except Exception:
            continue
    if not lons:
        raise RuntimeError(f"All transforms failed for {tile}")
    return (round(min(lons)-pad,6), round(min(lats)-pad,6),
            round(max(lons)+pad,6), round(max(lats)+pad,6))

def mgrs_to_bbox(tile, pad=0.05):
    try:
        bbox = _mgrs_bbox_via_package(tile, pad)
        print(f"   bbox via mgrs package  : {bbox}")
        return bbox
    except Exception:
        warnings.warn("mgrs package unavailable — using pyproj fallback.")
        bbox = _mgrs_bbox_via_pyproj(tile, pad)
        print(f"   bbox via pyproj fallback: {bbox}")
        return bbox

# ── Full tile bbox ────────────────────────────────────────────────────────────
print(f"\n🗺️  Resolving bbox for MGRS tile: {MGRS_TILE}")
BBOX_WGS84 = mgrs_to_bbox(MGRS_TILE)

if ROI is None:
    ROI_BBOX = BBOX_WGS84
    SEARCH_BBOX = ROI_BBOX
    from shapely.geometry import box as shp_box
    ROI_CENTROID = shp_box(ROI_BBOX[0], ROI_BBOX[1], ROI_BBOX[2], ROI_BBOX[3]).centroid
    ROI_CENTROID = (ROI_CENTROID.x, ROI_CENTROID.y)

# ── Derive target UTM EPSG ────────────────────────────────────────────────────
_zm         = re.match(r'^(\d{1,2})([C-HJ-NP-X])', MGRS_TILE.upper())
MGRS_ZONE   = int(_zm.group(1))    # 39
MGRS_BAND   = _zm.group(2)         # K
HEMI        = "N" if MGRS_LAT_BANDS.get(MGRS_BAND,0) >= 0 else "S"
TARGET_EPSG = (32600 if HEMI=="N" else 32700) + MGRS_ZONE   # 32739

# ── MODIS Sinusoidal CRS ──────────────────────────────────────────────────────
SINU_CRS = ("+proj=sinu +lon_0=0 +x_0=0 +y_0=0 "
            "+a=6371007.181 +b=6371007.181 +units=m +no_defs")

# ── Create output directories ─────────────────────────────────────────────────
for d in [DIR_OUT, DIR_HDF, DIR_TIFF, FIG_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Compute bbox dimensions ───────────────────────────────────────────────────
from pyproj import Transformer as _T
_tf   = _T.from_crs("EPSG:4326","EPSG:32739",always_xy=True)
_w, _s = _tf.transform(ROI_BBOX[0], ROI_BBOX[1])
_e, _n = _tf.transform(ROI_BBOX[2], ROI_BBOX[3])
_width_km  = abs((_e - _w) / 1000)
_height_km = abs((_n - _s) / 1000)

print(f"""
╔══════════════════════════════════════════════════════════╗
  📋  Configuration
╠══════════════════════════════════════════════════════════╣
  Site           : {SITE_NAME} (~{SITE_AREA_HA:,} ha)
  MGRS Tile      : {MGRS_TILE}
  Target CRS     : EPSG:{TARGET_EPSG}  (UTM Zone {MGRS_ZONE}{HEMI})
  Date Range     : {START_DATE}  →  {END_DATE}
  Products       : {[p[0] for p in PRODUCTS_TO_FETCH]}
  Output format  : {TIFF_FORMAT} GeoTIFF
╠══════════════════════════════════════════════════════════╣
  ROI BBox (WGS84):
    W : {ROI_BBOX[0]:.6f}°
    S : {ROI_BBOX[1]:.6f}°
    E : {ROI_BBOX[2]:.6f}°
    N : {ROI_BBOX[3]:.6f}°
    Width  : {_width_km:.2f} km
    Height : {_height_km:.2f} km
  Centroid       : {ROI_CENTROID[0]:.6f}°E,
                   {ROI_CENTROID[1]:.6f}°N
╠══════════════════════════════════════════════════════════╣
  OUTPUT dir     : {DIR_OUT}
  HDF dir        : {DIR_HDF}
  TIFF dir       : {DIR_TIFF}
  Figure dir     : {FIG_DIR}
╚══════════════════════════════════════════════════════════╝
""")

✅ Shapefile loaded: 1 features dissolved to 1 polygon
   Source CRS : EPSG:4326
   Working CRS: EPSG:4326 (WGS84)

🗺️  Resolving bbox for MGRS tile: 39KUA
   bbox via mgrs package  : (49.060189, -18.136395, 50.109642, -17.126057)

╔══════════════════════════════════════════════════════════╗
  📋  Configuration
╠══════════════════════════════════════════════════════════╣
  Site           : Betampona (~44,100 ha)
  MGRS Tile      : 39KUA
  Target CRS     : EPSG:32739  (UTM Zone 39S)
  Date Range     : 2016-01-01  →  2025-12-31
  Products       : ['MOD13Q1']
  Output format  : COG GeoTIFF
╠══════════════════════════════════════════════════════════╣
  ROI BBox (WGS84):
    W : 49.127100°
    S : -17.992170°
    E : 49.315780°
    N : -17.812470°
    Width  : 19.80 km
    Height : 20.08 km
  Centroid       : 49.221440°E,
                   -17.902320°N
╠══════════════════════════════════════════════════════════╣
  OUTPUT dir     : modis_veg_indices/39KUA
  HDF dir        : modis_veg_indices/

/tmp/ipykernel_949/3534674136.py:94: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  ROI_CENTROID = gdf_dissolved.geometry.centroid[0] # (49.215539, -17.903742)   # (lon, lat)


## Mapping the ROI

In [ ]:
# Cell 3b (Updated): Map using exact Betampona shapefile boundary
# 6-feature shapefile dissolved to single polygon, EPSG:32739 → WGS84

import matplotlib.pyplot      as plt
import matplotlib.patches     as mpatches
import matplotlib.patheffects as pe
import geopandas              as gpd
from pathlib          import Path
from pyproj           import Transformer

# ── Install contextily if needed ──────────────────────────────────────────────
try:
    import contextily as ctx
    HAS_CTX = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,"-m","pip",
                           "install","contextily","-q"])
    try:
        import contextily as ctx
        HAS_CTX = True
    except ImportError:
        HAS_CTX = False

CRS_PLOT = "EPSG:3857"   # Web Mercator

# ── Build GeoDataFrames ───────────────────────────────────────────────────────

try:
    roi_fname = "betampona.zip" # Replace with correct filename
    roi_feat = len(gdf_raw)
    
    # All 6 individual features (for display)
    gdf_features = gpd.read_file(SHP_PATH).to_crs(CRS_PLOT)
    
    # Dissolved single boundary
    gdf_roi = gpd.GeoDataFrame(
        {"label": [SITE_NAME]},
        geometry = [ROI],
        crs      = "EPSG:4326",
    ).to_crs(CRS_PLOT)
    
    # Search bbox
    gdf_bbox = gpd.GeoDataFrame(
        {"label": ["Search BBox"]},
        geometry = [box(*ROI_BBOX)],
        crs      = "EPSG:4326",
    ).to_crs(CRS_PLOT)
except:
    roi_fname = "MGRS Tile"
    roi_feat = 4
    source_crs = "4326"
    gdf_features = None

# MGRS tile
gdf_tile = gpd.GeoDataFrame(
    {"label": [f"MGRS {MGRS_TILE}"]},
    geometry = [box(*BBOX_WGS84)],
    crs      = "EPSG:4326",
).to_crs(CRS_PLOT)

if gdf_features is None:
    gdf_roi = gdf_tile
    gdf_bbox = gdf_tile

# ── Coordinate transformer ────────────────────────────────────────────────────
tf = Transformer.from_crs("EPSG:4326", CRS_PLOT, always_xy=True)

centroid_mx, centroid_my = tf.transform(*ROI_CENTROID)
# toamasina_mx, toamasina_my = tf.transform(49.4018, -18.1492)

# ── Bounds ────────────────────────────────────────────────────────────────────
tile_bounds = gdf_tile.total_bounds
bbox_bounds = gdf_bbox.total_bounds
roi_bounds  = gdf_roi.total_bounds

bbox_w_m = bbox_bounds[2] - bbox_bounds[0]
bbox_h_m = bbox_bounds[3] - bbox_bounds[1]

# Panel extents
left_cx, left_cy = centroid_mx, centroid_my
left_half        = 80_000   # ±80 km

left_xlim  = (left_cx - left_half, left_cx + left_half)
left_ylim  = (left_cy - left_half, left_cy + left_half)

pad_r      = max(bbox_w_m, bbox_h_m) * 0.40
right_xlim = (bbox_bounds[0]-pad_r,   bbox_bounds[2]+pad_r*1.8)
right_ylim = (bbox_bounds[1]-pad_r,   bbox_bounds[3]+pad_r)

# ─────────────────────────────────────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor("#f0f4f8")

# ═════════════════════════════════════════════════════════════════════════════
# LEFT PANEL — Regional context
# ═════════════════════════════════════════════════════════════════════════════
ax1.set_facecolor("#cce5ff")
ax1.set_xlim(*left_xlim)
ax1.set_ylim(*left_ylim)

if HAS_CTX:
    try:
        ctx.add_basemap(ax1, crs=CRS_PLOT, zoom=10,
                        source=ctx.providers.OpenStreetMap.Mapnik,
                        zorder=1)
    except Exception as e:
        print(f"⚠️  Left basemap: {e}")

# MGRS tile
gdf_tile.boundary.plot(ax=ax1, color="#e74c3c",
                        linewidth=1.5, linestyle="--", zorder=3)
gdf_tile.plot(ax=ax1, color="#e74c3c", alpha=0.04, zorder=2)

# Search bbox
gdf_bbox.plot(ax=ax1, color="#f39c12", alpha=0.18, zorder=4)
gdf_bbox.boundary.plot(ax=ax1, color="#f39c12",
                        linewidth=2.0, zorder=5)

if not gdf_features is None:
    # Individual shapefile features (light fill, thin outline)
    gdf_features.plot(ax=ax1, color="#27ae60", alpha=0.25, zorder=6)
    gdf_features.boundary.plot(ax=ax1, color="#27ae60",
                                 linewidth=0.8, linestyle=":", zorder=7)

# Dissolved ROI boundary — bold outline
gdf_roi.boundary.plot(ax=ax1, color="#1a7a40",
                       linewidth=3.0, zorder=8)

# Centroid
ax1.plot(centroid_mx, centroid_my,
         marker="*", color="#f1c40f", markersize=16,
         markeredgecolor="#1a7a40", markeredgewidth=1.2, zorder=10)

# Labels
ax1.annotate(
    f"  {SITE_NAME}",
    xy=(centroid_mx, centroid_my + 2500),
    fontsize=9, fontweight="bold", color="#1a7a40", zorder=11,
    path_effects=[pe.withStroke(linewidth=3, foreground="white")],
)
# ax1.plot(toamasina_mx, toamasina_my,
#          marker="o", color="#8e44ad", markersize=8,
#          markeredgecolor="white", markeredgewidth=0.8, zorder=10)
# ax1.annotate(
#     "  Toamasina",
#     xy=(toamasina_mx, toamasina_my),
#     fontsize=8, color="#8e44ad", zorder=11,
#     path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
# )

# Inset box
inset_rect = mpatches.Rectangle(
    (right_xlim[0], right_ylim[0]),
    right_xlim[1]-right_xlim[0],
    right_ylim[1]-right_ylim[0],
    linewidth=1.8, edgecolor="#c0392b",
    facecolor="none", linestyle=":", zorder=9,
)
ax1.add_patch(inset_rect)
ax1.annotate(
    "detail →",
    xy=(right_xlim[1], right_ylim[1]),
    fontsize=7.5, color="#c0392b", zorder=11,
    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
)

# Scale bar 20 km
sb_x0 = left_xlim[0] + (left_xlim[1]-left_xlim[0])*0.05
sb_y0 = left_ylim[0] + (left_ylim[1]-left_ylim[0])*0.04
ax1.plot([sb_x0, sb_x0+20_000], [sb_y0, sb_y0],
         color="black", linewidth=3, zorder=12, solid_capstyle="butt")
for x_ in [sb_x0, sb_x0+20_000]:
    ax1.plot([x_,x_],[sb_y0-500,sb_y0+500],
             color="black", linewidth=2, zorder=12)
ax1.text(sb_x0+10_000, sb_y0+1500, "20 km",
         ha="center", fontsize=7.5, zorder=12,
         path_effects=[pe.withStroke(linewidth=2, foreground="white")])

# North arrow
ax1.annotate("N", fontsize=13, fontweight="bold",
             xy=(0.96,0.94), xycoords="axes fraction",
             ha="center", va="bottom", color="#222222",
             path_effects=[pe.withStroke(linewidth=2.5,
                                         foreground="white")])
ax1.annotate("", xy=(0.96,0.93), xycoords="axes fraction",
             xytext=(0.96,0.84), textcoords="axes fraction",
             arrowprops=dict(arrowstyle="-|>",
                             color="#222222", lw=2.0))

legend1 = [
    mpatches.Patch(fc="#e74c3c", alpha=0.2, ec="#e74c3c",
                   ls="--", lw=1.5, label=f"MGRS Tile {MGRS_TILE}"),
    mpatches.Patch(fc="#f39c12", alpha=0.35, ec="#f39c12",
                   lw=2.0, label="Search BBox"),
    mpatches.Patch(fc="#27ae60", alpha=0.30, ec="#27ae60",
                   ls=":", lw=0.8, label="Shapefile features (6)"),
    mpatches.Patch(fc="none", ec="#1a7a40",
                   lw=3.0, label=f"{SITE_NAME} boundary (dissolved)"),
]
ax1.legend(handles=legend1, loc="lower left",
           fontsize=8, framealpha=0.92, edgecolor="#aaaaaa")

ax1.set_title(
    f"Regional Context — Central Maryland\n" # Replace with correct regional context
    f"MGRS Tile {MGRS_TILE}  ·  Centred on {SITE_NAME}",
    fontsize=11, fontweight="bold", pad=8,
)
ax1.set_xlabel("Easting (Web Mercator m)", fontsize=8)
ax1.set_ylabel("Northing (Web Mercator m)", fontsize=8)
ax1.tick_params(labelsize=7)

# ═════════════════════════════════════════════════════════════════════════════
# RIGHT PANEL — Zoomed ROI
# ═════════════════════════════════════════════════════════════════════════════
ax2.set_facecolor("#cce5ff")
ax2.set_xlim(*right_xlim)
ax2.set_ylim(*right_ylim)

if HAS_CTX:
    try:
        ctx.add_basemap(ax2, crs=CRS_PLOT, zoom=13,
                        source=ctx.providers.OpenStreetMap.Mapnik,
                        zorder=1)
    except Exception as e:
        print(f"⚠️  Right basemap: {e}")

# Search bbox
gdf_bbox.plot(ax=ax2, color="#f39c12", alpha=0.08, zorder=3)
gdf_bbox.boundary.plot(ax=ax2, color="#f39c12",
                        linewidth=2.5, zorder=4)

if not gdf_features is None:
    # Individual features — coloured by feature index
    colors = ["#1abc9c","#3498db","#9b59b6",
              "#e67e22","#e74c3c","#2ecc71"]
    for i, (_, row) in enumerate(gdf_features.iterrows()):
        feat_gdf = gpd.GeoDataFrame(
            geometry=[row.geometry], crs=CRS_PLOT
        )
        feat_gdf.plot(ax=ax2,
                      color=colors[i % len(colors)],
                      alpha=0.35, zorder=5)
        feat_gdf.boundary.plot(ax=ax2,
                                color=colors[i % len(colors)],
                                linewidth=1.0, zorder=6)

# Dissolved boundary — bold
gdf_roi.boundary.plot(ax=ax2, color="#1a7a40",
                       linewidth=3.5, zorder=7)

# Centroid star
ax2.plot(centroid_mx, centroid_my,
         marker="*", color="#f1c40f", markersize=20,
         markeredgecolor="#1a7a40", markeredgewidth=1.5, zorder=9)

# Site label
ax2.annotate(
    f"  {SITE_NAME}\n  ~{SITE_AREA_HA:,} ha",
    xy=(centroid_mx, centroid_my + bbox_h_m*0.12),
    fontsize=9, fontweight="bold", color="#1a7a40", zorder=10,
    path_effects=[pe.withStroke(linewidth=3, foreground="white")],
)

# Corner coordinate labels
corners = [
    (ROI_BBOX[0], ROI_BBOX[3], "NW", "left",  "bottom"),
    (ROI_BBOX[2], ROI_BBOX[3], "NE", "right", "bottom"),
    (ROI_BBOX[0], ROI_BBOX[1], "SW", "left",  "top"   ),
    (ROI_BBOX[2], ROI_BBOX[1], "SE", "right", "top"   ),
]
for lon, lat, lbl, ha, va in corners:
    mx, my = tf.transform(lon, lat)
    ax2.plot(mx, my, marker="+", color="#f39c12",
             markersize=12, markeredgewidth=2.0, zorder=8)
    ax2.annotate(
        f"{lbl}: {lon:.4f}°E\n      {abs(lat):.4f}°S",
        xy=(mx, my), fontsize=6.5, ha=ha, va=va,
        color="#333333", zorder=10,
        path_effects=[pe.withStroke(linewidth=2, foreground="white")],
    )

# Dimension arrows
roi_w_km = bbox_w_m / 1000
roi_h_km = bbox_h_m / 1000

y_arr = bbox_bounds[1] - bbox_h_m*0.10
ax2.annotate("",
    xy=(bbox_bounds[2], y_arr),
    xytext=(bbox_bounds[0], y_arr),
    arrowprops=dict(arrowstyle="<->", color="#f39c12", lw=1.8),
    zorder=7)
ax2.text((bbox_bounds[0]+bbox_bounds[2])/2, y_arr,
         f"  {roi_w_km:.1f} km  ",
         ha="center", va="top", fontsize=8.5,
         color="#e67e22", fontweight="bold",
         path_effects=[pe.withStroke(linewidth=2.5,
                                     foreground="white")],
         zorder=8)

x_arr = bbox_bounds[2] + bbox_w_m*0.10
ax2.annotate("",
    xy=(x_arr, bbox_bounds[3]),
    xytext=(x_arr, bbox_bounds[1]),
    arrowprops=dict(arrowstyle="<->", color="#f39c12", lw=1.8),
    zorder=7)
ax2.text(x_arr, (bbox_bounds[1]+bbox_bounds[3])/2,
         f"  {roi_h_km:.1f} km",
         ha="left", va="center", fontsize=8.5,
         color="#e67e22", fontweight="bold",
         path_effects=[pe.withStroke(linewidth=2.5,
                                     foreground="white")],
         zorder=8)

# Scale bar 2 km
sb2_x0 = right_xlim[0] + (right_xlim[1]-right_xlim[0])*0.05
sb2_y0 = right_ylim[0] + (right_ylim[1]-right_ylim[0])*0.04
ax2.plot([sb2_x0, sb2_x0+2_000], [sb2_y0, sb2_y0],
         color="black", linewidth=3, zorder=12,
         solid_capstyle="butt")
for x_ in [sb2_x0, sb2_x0+2_000]:
    ax2.plot([x_,x_],
             [sb2_y0-bbox_h_m*0.008, sb2_y0+bbox_h_m*0.008],
             color="black", linewidth=2, zorder=12)
ax2.text(sb2_x0+1_000, sb2_y0+bbox_h_m*0.018,
         "2 km", ha="center", fontsize=7.5, zorder=12,
         path_effects=[pe.withStroke(linewidth=2, foreground="white")])

# North arrow
ax2.annotate("N", fontsize=13, fontweight="bold",
             xy=(0.96,0.94), xycoords="axes fraction",
             ha="center", va="bottom", color="#222222",
             path_effects=[pe.withStroke(linewidth=2.5,
                                         foreground="white")])
ax2.annotate("", xy=(0.96,0.93), xycoords="axes fraction",
             xytext=(0.96,0.84), textcoords="axes fraction",
             arrowprops=dict(arrowstyle="-|>",
                             color="#222222", lw=2.0))

legend2 = [
    mpatches.Patch(fc="#f39c12", alpha=0.25, ec="#e67e22",
                   lw=2.5,
                   label=f"Search BBox ({roi_w_km:.1f}×{roi_h_km:.1f} km)"),
    mpatches.Patch(fc="#27ae60", alpha=0.35, ec="#1a7a40",
                   lw=3.5,
                   label=f"{SITE_NAME} boundary (~{SITE_AREA_HA:,} ha)"),
    mpatches.Patch(fc="#aaaaaa", alpha=0.35, ec="#aaaaaa",
                   lw=1.0, label="Individual shapefile features (6)"),
]
ax2.legend(handles=legend2, loc="upper right",
           fontsize=8, framealpha=0.92, edgecolor="#aaaaaa")

ax2.set_title(
    f"Zoomed — {SITE_NAME}  (EPSG:{source_crs} → WGS84)\n"
    f"W:{ROI_BBOX[0]:.4f}° E:{ROI_BBOX[2]:.4f}°  |  "
    f"S:{ROI_BBOX[1]:.4f}° N:{ROI_BBOX[3]:.4f}°",
    fontsize=10, fontweight="bold", pad=8,
)
ax2.set_xlabel("Easting (Web Mercator m)", fontsize=8)
ax2.set_ylabel("Northing (Web Mercator m)", fontsize=8)
ax2.tick_params(labelsize=7)

# ── Overall title ─────────────────────────────────────────────────────────────
fig.suptitle(
    f"{SITE_NAME} — Boundary & Search BBox\n"
    f"MGRS {MGRS_TILE}  ·  EPSG:{TARGET_EPSG}  ·  "
    f"Source: {roi_fname} ({roi_feat} features, EPSG:{source_crs})",
    fontsize=12, fontweight="bold", y=1.01, color="#1a1a2e",
)

plt.tight_layout()

# map_path = Path(FIG_DIR) / f"{SITE_NAME}_boundary_map_{MGRS_TILE}.png"
# Path(FIG_DIR).mkdir(parents=True, exist_ok=True)
# plt.savefig(map_path, dpi=200, bbox_inches="tight",
#             facecolor=fig.get_facecolor())
plt.show()
# print(f"\n✅ Map saved → {map_path}")

## Grabbing the HDF files

In [5]:
import earthaccess
from pathlib import Path
from tqdm    import tqdm

# Suppress warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

HDF4_MAGIC = b"\x0e\x03\x13\x01"

def is_valid_hdf(fpath: Path) -> bool:
    if not fpath.exists() or fpath.stat().st_size < 100_000:
        return False
    with open(fpath, "rb") as fh:
        return fh.read(4) == HDF4_MAGIC

downloaded_files = {}

for short_name, version in PRODUCTS_TO_FETCH:

    print(f"\n{'─'*55}")
    print(f"🔎 {short_name} v{version}")
    print(f"{'─'*55}")

    results = earthaccess.search_data(
        short_name   = short_name,
        version      = version,
        bounding_box = SEARCH_BBOX,
        temporal     = (START_DATE, END_DATE),
        count        = -1,
    )
    print(f"   CMR results  : {len(results)} granules")

    if not results:
        print(f"   ⚠️  No granules found.")
        downloaded_files[short_name] = []
        continue

    # Filter to granules with HDF links
    hdf_granules = []
    for granule in results:
        try:
            links = (granule.data_links(access="external") +
                     granule.data_links(access="on_prem"))
        except Exception:
            links = [u["URL"] for u in
                     granule.get("umm",{}).get("RelatedUrls",[])
                     if "URL" in u]
        if any(Path(lnk).suffix.lower() == ".hdf" for lnk in links):
            hdf_granules.append(granule)

    print(f"   HDF granules : {len(hdf_granules)}")

    product_dir = Path(DIR_HDF) / short_name
    product_dir.mkdir(parents=True, exist_ok=True)

    raw_files = earthaccess.download(
        hdf_granules, local_path=str(product_dir)
    )

    # Verify — remove non-HDF files
    valid_hdfs, removed = [], 0
    for f in raw_files:
        fpath = Path(f)
        if is_valid_hdf(fpath):
            valid_hdfs.append(fpath)
        else:
            fpath.unlink(missing_ok=True)
            removed += 1

    downloaded_files[short_name] = valid_hdfs
    print(f"   Valid HDF    : {len(valid_hdfs)}")
    if removed:
        print(f"   Removed      : {removed} non-HDF files")
    print(f"   Saved to     : {product_dir}")

print(f"\n✅ Download complete.")


───────────────────────────────────────────────────────
🔎 MOD13Q1 v061
───────────────────────────────────────────────────────
   CMR results  : 232 granules
   HDF granules : 232


QUEUEING TASKS | :   0%|          | 0/661 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/661 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/661 [00:00<?, ?it/s]

   Valid HDF    : 232
   Removed      : 429 non-HDF files
   Saved to     : modis_veg_indices/39KUA/modis_hdf/MOD13Q1

✅ Download complete.


## 3. Clean data and convert HDF to TIFF
This section parses the HDF files to actually get the desired bands (NDVI, EVI, QA, DOY, RED, NIR, BLUE). It also cleans the data to ensure there are no fill values or values outside the valid range, as well as doing some QA masking to remove clouds, snow, ice, and water. The QA masking also removes data that is not good quality or useful. There shouldn't be anything that needs to be changed in this section, unless the masking seems too strict or not strict enough, in which case just change the mask_qa_values() function. Note that if you want to replace files after making any changes to the processing you will have to comment out the lines that check if the file already exists and just grabs it.

In [19]:
import rioxarray
import rasterio
import numpy    as np
import pandas   as pd
import subprocess, re, warnings
from pathlib          import Path
from tqdm             import tqdm
from rasterio.enums   import Resampling as RasterioResampling
from rasterio.shutil  import copy as rio_copy

try:
    from rioxarray.enums import Resampling
except ImportError:
    from rasterio.enums  import Resampling

# ── Layer configuration — includes DOY for all composite products ─────────────
LAYER_CONFIG = {
    "MOD13Q1": {
        "layers": {
            "QA"  : "250m 16 days VI Quality",
            "NDVI": "250m 16 days NDVI",
            "EVI" : "250m 16 days EVI",
            "DOY" : "250m 16 days composite day of the year",
        },
        "scale"  : {"NDVI": 0.0001, "EVI": 0.0001},
        "valid"  : {
            "NDVI": (-2000, 10000),
            "EVI" : (-2000, 10000),
            "DOY" : (1, 366),
        },
        "nodata" : -3000,
        "dtype"  : {
            "QA"  : "uint16",
            "NDVI": "float32",
            "EVI" : "float32",
            "DOY" : "int16",
        },
    },
    "MYD13Q1": {
        "layers": {
            "QA"  : "250m 16 days VI Quality",
            "NDVI": "250m 16 days NDVI",
            "EVI" : "250m 16 days EVI",
            "DOY" : "250m 16 days composite day of the year",
        },
        "scale"  : {"NDVI": 0.0001, "EVI": 0.0001},
        "valid"  : {
            "NDVI": (-2000, 10000),
            "EVI" : (-2000, 10000),
            "DOY" : (1, 366),
        },
        "nodata" : -3000,
        "dtype"  : {
            "QA"  : "uint16",
            "NDVI": "float32",
            "EVI" : "float32",
            "DOY" : "int16",
        },
    },
    "MOD13A2": {
        "layers": {
            "QA"  : "1 km 16 days VI Quality",
            "NDVI": "1 km 16 days NDVI",
            "EVI" : "1 km 16 days EVI",
            "DOY" : "1 km 16 days composite day of the year",
        },
        "scale"  : {"NDVI": 0.0001, "EVI": 0.0001},
        "valid"  : {
            "NDVI": (-2000, 10000),
            "EVI" : (-2000, 10000),
            "DOY" : (1, 366),
        },
        "nodata" : -3000,
        "dtype"  : {
            "QA"  : "uint16",
            "NDVI": "float32",
            "EVI" : "float32",
            "DOY" : "int16",
        },
    },
    "MYD13A2": {
        "layers": {
            "QA"  : "1 km 16 days VI Quality",
            "NDVI": "1 km 16 days NDVI",
            "EVI" : "1 km 16 days EVI",
            "DOY" : "1 km 16 days composite day of the year",
        },
        "scale"  : {"NDVI": 0.0001, "EVI": 0.0001},
        "valid"  : {
            "NDVI": (-2000, 10000),
            "EVI" : (-2000, 10000),
            "DOY" : (1, 366),
        },
        "nodata" : -3000,
        "dtype"  : {
            "QA"  : "uint16",
            "NDVI": "float32",
            "EVI" : "float32",
            "DOY" : "int16",
        },
    },
    "MOD09GA": {
        "layers": {
            "Red" : "sur_refl_b01_1",
            "NIR" : "sur_refl_b02_1",
            "Blue": "sur_refl_b03_1",
        },
        "scale"  : {"Red": 0.0001, "NIR": 0.0001, "Blue": 0.0001},
        "valid"  : {
            "Red" : (-100, 16000),
            "NIR" : (-100, 16000),
            "Blue": (-100, 16000),
        },
        "nodata" : -28672,
        "dtype"  : {
            "Red" : "float32",
            "NIR" : "float32",
            "Blue": "float32",
        },
    },
    "MYD09GA": {
        "layers": {
            "Red" : "sur_refl_b01_1",
            "NIR" : "sur_refl_b02_1",
            "Blue": "sur_refl_b03_1",
        },
        "scale"  : {"Red": 0.0001, "NIR": 0.0001, "Blue": 0.0001},
        "valid"  : {
            "Red" : (-100, 16000),
            "NIR" : (-100, 16000),
            "Blue": (-100, 16000),
        },
        "nodata" : -28672,
        "dtype"  : {
            "Red" : "float32",
            "NIR" : "float32",
            "Blue": "float32",
        },
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────────────────────────────────────

def get_sds_path(hdf_path: Path, layer_substr: str) -> str:
    result = subprocess.run(
        ["gdalinfo", str(hdf_path)], capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if "SUBDATASET" in line and "_NAME=" in line and layer_substr in line:
            return line.split("=",1)[1].strip()
    available = [
        line.split("=",1)[1].strip()
        for line in result.stdout.splitlines()
        if "SUBDATASET" in line and "_NAME=" in line
    ]
    raise ValueError(
        f"Layer '{layer_substr}' not found in {hdf_path.name}.\n"
        f"Available:\n" + "\n".join(f"  {s}" for s in available)
    )

def parse_modis_date(fname: str) -> pd.Timestamp:
    m = re.search(r'\.A(\d{4})(\d{3})\.', fname)
    if not m:
        m = re.search(r'A(\d{4})(\d{3})', fname)
    if not m:
        raise ValueError(f"Cannot parse date from: {fname}")
    return (pd.Timestamp(f"{m.group(1)}-01-01")
            + pd.Timedelta(days=int(m.group(2))-1))

def fix_y_axis(da):
    """Ensure y runs N→S (decreasing). Flip if S→N (increasing)."""
    if len(da.y) > 1 and da.y.values[-1] > da.y.values[0]:
        da = da.isel(y=slice(None,None,-1))
    return da

def mask_fill_values(da, nodata_raw, valid_range=None):
    """Mask nodata, out-of-range, and overflow values before cast."""
    da = da.where(da != nodata_raw)
    if valid_range is not None:
        vmin, vmax = valid_range
        da = da.where((da >= vmin) & (da <= vmax))
    da = da.where((da > -1e10) & (da < 1e10))
    return da

def mask_qa_values(da, qa):
    """Mask clouds, snow, ice, and water values before cast."""
    adjacent_cloud = ((qa >> 8) & 1) == 1
    mixed_cloud = ((qa >> 10) & 1) == 1
    snow_ice = ((qa >> 14) & 1) == 1
    land_water = (qa >> 11) & 0b111
    land = land_water == 1
    vi_quality = qa & 0b11
    good = vi_quality == 0
    usefulness = (qa >> 2) & 0b1111
    useful = usefulness <= 2
    mask = (
        good &
        useful &
        land &
        ~adjacent_cloud &
        ~mixed_cloud &
        ~snow_ice
    )
    da = da.where(mask)
    return da

def safe_cast(arr: np.ndarray, dtype: str) -> np.ndarray:
    """Clip to dtype range and cast — no RuntimeWarning."""
    if dtype == "float32":
        f32_min = float(np.finfo(np.float32).min)
        f32_max = float(np.finfo(np.float32).max)
        arr_out = np.where(
            np.isfinite(arr),
            np.clip(arr, f32_min, f32_max),
            np.nan
        )
        return arr_out.astype(np.float32)
    elif dtype == "uint16":
        arr_out = np.where(np.isnan(arr.astype(float)), 65535.0, arr)
        return np.clip(arr_out, 0, 65534).astype(np.uint16)
    elif dtype == "int16":
        # DOY: valid 1–366, nodata sentinel = -1
        arr_out = np.where(np.isnan(arr.astype(float)), -1.0, arr)
        return np.clip(arr_out, -1, 366).astype(np.int16)
    else:
        return arr.astype(dtype)

def build_tiff_profile(da, dtype: str, nodata_val):
    """Build rasterio profile. Returns (profile, da) — da may be y-flipped."""
    transform = da.rio.transform()
    if transform.e > 0:
        da        = da.isel(y=slice(None,None,-1))
        transform = da.rio.transform()
    predictor = 3 if np.issubdtype(np.dtype(dtype), np.floating) else 2
    profile = {
        "driver"    : "GTiff",
        "dtype"     : dtype,
        "width"     : da.shape[-1],
        "height"    : da.shape[-2],
        "count"     : 1,
        "crs"       : da.rio.crs,
        "transform" : transform,
        "nodata"    : nodata_val,
        "compress"  : "deflate",
        "predictor" : predictor,
        "zlevel"    : 6,
        "tiled"     : True,
        "blockxsize": 512,
        "blockysize": 512,
    }
    return profile, da

def write_geotiff(da, out_path: Path, dtype: str,
                  nodata_val, tags: dict, tiff_format: str):
    """
    Write DataArray to COG or standard GeoTIFF.
    copy_src_overviews passed ONLY to rio_copy — never in profile dict.
    """
    profile, da = build_tiff_profile(da, dtype, nodata_val)
    arr         = safe_cast(da.values, dtype)

    if tiff_format == "COG":
        tmp_path = out_path.with_suffix(".tmp.tif")
        with rasterio.open(tmp_path, "w", **profile) as dst:
            dst.write(arr, 1)
            dst.update_tags(**tags)
        with rasterio.open(tmp_path, "r+") as tmp:
            tmp.build_overviews([2,4,8,16,32], RasterioResampling.average)
            tmp.update_tags(ns="rio_overview", resampling="average")
        rio_copy(tmp_path, out_path,
                 copy_src_overviews=True, **profile)   # ← only here
        tmp_path.unlink(missing_ok=True)
    else:
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(arr, 1)
            dst.update_tags(**tags)

def hdf_to_geotiff(hdf_path: Path, short_name: str,
                   target_epsg: int, out_dir: Path,
                   tiff_format: str = "COG") -> dict:
    """
    Convert one MODIS HDF4 → one GeoTIFF per band.
    Returns {band_name: Path}. Skips existing files.
    File naming: {short_name}_{band}_{YYYYMMDD}.tif
    """
    config    = LAYER_CONFIG[short_name]
    date      = parse_modis_date(hdf_path.name)
    date_tag  = date.strftime("%Y%m%d")
    out_paths = {}

    # Store QA to mask
    qa = None
    
    for band_name, layer_key in config["layers"].items():

        out_tif = out_dir / f"{short_name}_{band_name}_{date_tag}.tif"

        if out_tif.exists():
            out_paths[band_name] = out_tif
            
            # Check pixel count in search bbox
            from rasterio.windows import from_bounds
            
            with rasterio.open(out_tif) as src:
                window = from_bounds(
                    SEARCH_BBOX[0], SEARCH_BBOX[1], SEARCH_BBOX[2], SEARCH_BBOX[3],
                    transform=src.transform
                )
                pixel_count = window.width * window.height
                print(pixel_count)
     
            continue

        try:
            sds = get_sds_path(hdf_path, layer_key)
            da  = rioxarray.open_rasterio(
                      sds, masked=True, lock=False
                  ).squeeze("band", drop=True)
            da = da.rio.write_crs(SINU_CRS)
            da = fix_y_axis(da)
           
            # Grab qa mask
            if band_name == 'QA':
                qa = da.copy()
                # Should be uint16
                qa = qa.astype(config['dtype']['QA'])
            
            da = mask_fill_values(
                da,
                nodata_raw  = config["nodata"],
                valid_range = config["valid"].get(band_name),
            )
            
            # Apply qa mask if relevant
            if not qa is None and (band_name == 'NDVI' or band_name == 'EVI'):
                da = mask_qa_values(da, qa)
                
            da = da.rio.reproject(
                f"EPSG:{target_epsg}",
                resampling = Resampling.bilinear,
                nodata     = np.nan,
            )
            da = fix_y_axis(da)

            if band_name in config["scale"]:
                da = da * config["scale"][band_name]

            dtype = config["dtype"].get(band_name, "float32")
            if dtype == "uint16":
                nodata_v = np.uint16(65535)
            elif dtype == "int16":
                nodata_v = np.int16(-1)
            else:
                nodata_v = np.float32(np.nan)

            tags = {
                "MODIS_product"   : short_name,
                "band"            : band_name,
                "acquisition_date": date_tag,
                "source_hdf"      : hdf_path.name,
                "crs"             : f"EPSG:{target_epsg}",
                "utm_zone"        : f"{MGRS_ZONE}{HEMI}",
                "mgrs_tile"       : MGRS_TILE,
                "scale_applied"   : str(config["scale"].get(band_name,"none")),
            }

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                write_geotiff(da, out_tif, dtype, nodata_v,
                              tags, tiff_format)

            out_paths[band_name] = out_tif

        except Exception as e:
            print(f"\n     ⚠️  {band_name} [{hdf_path.name}]: "
                  f"{type(e).__name__}: {e}")

    return out_paths

# ── Conversion loop ───────────────────────────────────────────────────────────
converted_files = {}

for short_name, hdf_files in downloaded_files.items():

    if short_name not in LAYER_CONFIG:
        print(f"⚠️  No layer config for '{short_name}' — skipping.")
        continue

    out_dir = Path(DIR_TIFF) / short_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'─'*55}")
    print(f"🔄 {short_name}: {len(hdf_files)} HDF → {TIFF_FORMAT} GeoTIFF")
    print(f"   Output : {out_dir}")
    print(f"{'─'*55}")

    band_paths = {b: [] for b in LAYER_CONFIG[short_name]["layers"]}
    failed     = []

    for hdf_file in tqdm(sorted(hdf_files), desc=f"  {short_name}"):
        try:
            result = hdf_to_geotiff(
                hdf_path    = Path(hdf_file),
                short_name  = short_name,
                target_epsg = TARGET_EPSG,
                out_dir     = out_dir,
                tiff_format = TIFF_FORMAT,
            )
            for band_name, tif_path in result.items():
                band_paths[band_name].append(tif_path)
        except Exception as e:
            failed.append(Path(hdf_file).name)
            print(f"\n   ❌ {Path(hdf_file).name}: {e}")

    converted_files[short_name] = band_paths

    print()
    for band_name, paths in band_paths.items():
        print(f"   ✅ {band_name:6s}: {len(paths)} GeoTIFFs")
    if failed:
        print(f"   ❌ Failed : {len(failed)}")
        for f in failed[:5]:
            print(f"      • {f}")

    # Verify first output
    sample_band = next(iter(band_paths))
    if band_paths[sample_band]:
        sample = band_paths[sample_band][0]
        with rasterio.open(sample) as src:
            t = src.transform
            print(f"\n   📋 Verification — {sample.name}")
            print(f"      CRS     : {src.crs}")
            print(f"      Shape   : {src.height} × {src.width}")
            print(f"      Dtype   : {src.dtypes[0]}")
            print(f"      Nodata  : {src.nodata}")
            print(f"      X pixel : {t.a:.2f} m "
                  f"{'✅' if t.a > 0 else '❌ x flipped'}")
            print(f"      Y pixel : {t.e:.2f} m "
                  f"{'✅' if t.e < 0 else '❌ y flipped'}")
            print(f"      Bounds  : {src.bounds}")
            print(f"      Size    : {sample.stat().st_size/1e6:.2f} MB")

print("\n✅ TIFF download complete.")


───────────────────────────────────────────────────────
🔄 MOD13Q1: 232 HDF → COG GeoTIFF
   Output : modis_veg_indices/39KUA/modis_tiff/MOD13Q1
───────────────────────────────────────────────────────


  MOD13Q1:   2%|▏         | 5/232 [00:00<00:04, 47.66it/s]

5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07


  MOD13Q1:   4%|▍         | 10/232 [00:00<00:04, 48.15it/s]

5.076909340540879e-07


...

5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07


  MOD13Q1:  99%|█████████▊| 229/232 [00:04<00:00, 47.80it/s]

5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07


  MOD13Q1: 100%|██████████| 232/232 [00:04<00:00, 48.24it/s]

5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07
5.076909340540879e-07

   ✅ QA    : 232 GeoTIFFs
   ✅ NDVI  : 232 GeoTIFFs
   ✅ EVI   : 232 GeoTIFFs
   ✅ DOY   : 232 GeoTIFFs

   📋 Verification — MOD13Q1_QA_20151219.tif
      CRS     : EPSG:32739
      Shape   : 4367 × 5321
      Dtype   : uint16
      Nodata  : 65535.0
      X pixel : 258.43 m ✅
      Y pixel : -258.43 m ✅
      Bounds  : BoundingBox(left=-643858.9368844796, bottom=7766029.717506894, right=731229.1782711917, top=8894578.824607968)
      Size    : 8.05 MB

✅ TIFF download complete.


## 4. MODIS and HLS Comparision

The following code compiles the data from MODIS and HLS into their own respective DataFrames that contain a row for each composite as well as the vegatation index mean, median, min, max, standard deviation, and valid pixels. These DataFrames are then used to create figures comparing the two, including a vegetation index time series, a graph of the vegetation index multi-year seasonal means, and an annual phenology summary that shows the annual min, max, start of season (calculated as the first DOY to reach the greenup threshold, 15% amplitude), end of season (first DOY to fall below threshold after peak), and season length.

In [ ]:
import os
import re
import gc
import math
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from rasterio.mask import mask as rio_mask
from shapely.ops import unary_union, linemerge, polygonize

output_dir = f"hls_veg_indices/{MGRS_TILE}"
veg_index = "ndvi" #"evi"

In [ ]:
all_composites_hls = []
all_composites_modis = []

In [ ]:
def compute_hls_vi_stats(veg_index, data_dir, outdir,
                         tile_id, start_year=None, end_year=None,
                         roi_shp=None, roi_name=None):
    vi_pattern = re.compile(
        rf'^{re.escape(tile_id)}_10day_median_(\d{{7}})\_{veg_index.upper()}.tif$',
        re.IGNORECASE
    )
    doy_pattern = re.compile(
        rf'^{re.escape(tile_id)}_DOY_10day_(\d{{7}})\.tif$',
        re.IGNORECASE
    )

    vi_files  = {}
    doy_files = {}

    for fname in os.listdir(data_dir):
        m = vi_pattern.match(fname)
        if m:
            vi_files[m.group(1)] = os.path.join(data_dir, fname)
            continue
        m = doy_pattern.match(fname)
        if m:
            doy_files[m.group(1)] = os.path.join(data_dir, fname)

    years = sorted(set(k[:4] for k in vi_files))
    years = [y for y in years if
             (start_year is None or int(y) >= start_year) and
             (end_year   is None or int(y) <= end_year)]

    if not years:
        print(f"No data found for year range {start_year}–{end_year}")
        return pd.DataFrame()

    print(f"Processing {len(years)} year(s): {years[0]}–{years[-1]}")
    records = []
    
    roi = None
    if roi_shp is not None:
        if Path(roi_shp).suffix == ".zip":
            roi = gpd.read_file(f"zip://{roi_shp}")
        else:
            roi = gpd.read_file(roi_shp)
        print(f"ROI loaded: {roi_shp}  ({len(roi)} feature(s))")
        if "LineString" in roi.geom_type.values:
            print("Shapefile contains LineStrings, merging into a Polygon.")
            # Create a polygon from the LineStrings
            boundary = linemerge(unary_union(roi.geometry))
            polys = list(polygonize(boundary))
            
            # Redefine the roi using the polygon
            roi = gpd.GeoDataFrame(
                geometry=polys,
                crs=roi.crs
            )

    for year in years:
        year_keys       = sorted(k for k in vi_files if k[:4] == year)
        print(f"year_keys, {year_keys}")
        annual_vi_stack = []

        for doy_key in year_keys:
            vi_path  = vi_files[doy_key]
            doy_path = doy_files.get(doy_key)
            year_int   = int(doy_key[:4])
            doy_start  = int(doy_key[4:])
            start_date = datetime(year_int, 1, 1) + timedelta(days=doy_start - 1)

            vi = rxr.open_rasterio(vi_path, masked=True).squeeze("band", drop=True)
            vi.attrs.pop("scale_factor", None)
            vi.attrs.pop("add_offset",   None)

            roi_reproj = None
            if roi is not None:
                roi_reproj = roi.to_crs(vi.rio.crs)
                vi = vi.rio.clip(roi_reproj.geometry, roi_reproj.crs, drop=True)

            if doy_path:
                doy_da = rxr.open_rasterio(doy_path, masked=True).squeeze("band", drop=True)
                doy_da = doy_da.where(vi.notnull())
                doy_da = doy_da.where(doy_da <= 9)

                mean_doy_offset = float(doy_da.mean(skipna=True).values)
                if np.isnan(mean_doy_offset):
                    print(f"Warning: No valid DOY pixels for {doy_key}, using start date")
                    mean_doy_offset = 0.0

                rep_date = start_date + timedelta(days=round(mean_doy_offset))
            else:
                print(f"Warning: No DOY tif for {doy_key}, using start date")
                mean_doy_offset = 0.0
                rep_date        = start_date

            valid = vi.values[~np.isnan(vi.values)]
            if valid.size == 0:
                print(f"Warning: No valid VI pixels for {doy_key} — skipping")
                del vi
                gc.collect()
                continue

            try:
                all_composites_hls.append(vi.assign_coords(time=rep_date).expand_dims("time"))
            except UnboundLocalError:
                pass

            records.append({
                "doy_key":             doy_key,
                "composite_start":     start_date,
                "composite_mean_date": rep_date,
                "mean_doy_offset":     round(mean_doy_offset, 2),
                f"{veg_index}_mean":   float(np.mean(valid)),
                f"{veg_index}_median": float(np.median(valid)),
                f"{veg_index}_min":    float(np.min(valid)),
                f"{veg_index}_max":    float(np.max(valid)),
                f"{veg_index}_stdev":  float(np.std(valid, ddof=1)),
                "n_valid_pixels":      int(valid.size),
            })

            annual_vi_stack.append(vi.assign_coords(time=rep_date).expand_dims("time"))

        if not annual_vi_stack:
            print(f"No valid composites for {year} — skipping annual tif")
            continue

        stacked     = xr.concat(annual_vi_stack, dim="time")
        annual_mean = stacked.mean(dim="time", skipna=True)
        annual_std  = stacked.std( dim="time", skipna=True, ddof=1)

        annual_2band = xr.concat(
            [annual_mean.expand_dims("band"),
             annual_std.expand_dims("band")],
            dim="band"
        ).assign_coords(band=[1, 2])

        annual_2band.attrs = {
            "long_name":    [f"{veg_index.upper()}_annual_mean", f"{veg_index.upper()}_annual_stdev"],
            "year":         year,
            "n_composites": len(annual_vi_stack),
        }
        vi_crs = annual_vi_stack[0].rio.crs
        annual_2band = annual_2band.rio.write_crs(vi_crs)

        annual_fname = f"{tile_id}_{veg_index.upper()}_annual_{year}_{veg_index.upper()}.tif"
        annual_path  = os.path.join(output_dir, annual_fname)
        annual_2band.rio.to_raster(annual_path, compress="lzw")
        print(f"Annual written: {annual_fname}  |  n_composites={len(annual_vi_stack)}")

        del stacked, annual_mean, annual_std, annual_2band, annual_vi_stack
        try:
            del vi
        except UnboundLocalError:
            pass
        gc.collect()

    df = pd.DataFrame(records).sort_values("composite_start").reset_index(drop=True)
    if roi_name:
        csv_path = os.path.join(output_dir, f"HLS_{tile_id}_{veg_index.upper()}_timeseries_stats-{roi_name}.csv")
    else:
        csv_path = os.path.join(output_dir, f"HLS_{tile_id}_{veg_index.upper()}_timeseries_stats.csv")
    df.to_csv(csv_path, index=False)
    print(f"Timeseries CSV: {csv_path}")
    return df

In [ ]:
def compute_modis_vi_stats(veg_index, data_dir, outdir,
                           tile_id, start_year=None, end_year=None,
                           hls_data_dir=None, roi_shp=None, roi_name=None):

    vi_pattern = re.compile(
        rf'^MOD13Q1_{re.escape(veg_index.upper())}_(\d{{8}})\.tif$',
        re.IGNORECASE
    )
    doy_pattern = re.compile(
        r'^MOD13Q1_DOY_(\d{8})\.tif$',
        re.IGNORECASE
    )

    vi_files  = {}
    doy_files = {}

    for fname in os.listdir(data_dir):
        m = vi_pattern.match(fname)
        if m:
            vi_files[m.group(1)] = os.path.join(data_dir, fname)
            continue
        m = doy_pattern.match(fname)
        if m:
            doy_files[m.group(1)] = os.path.join(data_dir, fname)

    # Filter by year range on the YYYYMMDD key
    vi_files = {k: v for k, v in vi_files.items() if
                (start_year is None or int(k[:4]) >= start_year) and
                (end_year   is None or int(k[:4]) <= end_year)}

    if not vi_files:
        print(f"No data found for year range {start_year}–{end_year}")
        return pd.DataFrame()

    years = sorted(set(k[:4] for k in vi_files))
    years = [y for y in years if
             (start_year is None or int(y) >= start_year) and
             (end_year   is None or int(y) <= end_year)]

    print(f"Processing {len(years)} year(s): {years[0]}–{years[-1]}")

    # Build HLS bounding box to mask to MGRS tile (MODIS has larger footprint)
    hls_bbox_gdf = None
    if hls_data_dir is not None:
        hls_tifs = [
            os.path.join(hls_data_dir, f)
            for f in os.listdir(hls_data_dir)
            if f.lower().endswith(".tif")
        ]
        if hls_tifs:
            sample = rxr.open_rasterio(hls_tifs[0], masked=True)
            hls_crs    = sample.rio.crs
            hls_bounds = sample.rio.bounds()   # (left, bottom, right, top)
            del sample
            gc.collect()

            hls_bbox_geom = box(*hls_bounds)             
            hls_bbox_gdf  = gpd.GeoDataFrame(
                geometry=[hls_bbox_geom], crs=hls_crs
            )
            print(
                f"HLS tile bounds loaded from: {os.path.basename(hls_tifs[0])}\n"
                f"  CRS   : {hls_crs}\n"
                f"  Bounds: left={hls_bounds[0]:.2f}, bottom={hls_bounds[1]:.2f}, "
                f"right={hls_bounds[2]:.2f}, top={hls_bounds[3]:.2f}"
            )
        else:
            print("Warning: hls_data_dir provided but no .tif files found — skipping HLS bounds mask")

    roi = None
    if roi_shp is not None:
        if Path(roi_shp).suffix == ".zip":
            roi = gpd.read_file(f"zip://{roi_shp}")
        else:
            roi = gpd.read_file(roi_shp)
        print(f"ROI loaded: {roi_shp}  ({len(roi)} feature(s))")
        if "LineString" in roi.geom_type.values:
            print("Shapefile contains LineStrings, merging into a Polygon.")
            # Create a polygon from the LineStrings
            boundary = linemerge(unary_union(roi.geometry))
            polys = list(polygonize(boundary))
            
            # Redefine the roi using the polygon
            roi = gpd.GeoDataFrame(
                geometry=polys,
                crs=roi.crs
            )

    records = []

    for year in years:
        print(year)
        annual_vi_stack = []
        year_keys = sorted(k for k in vi_files if k[:4] == year)

        for date_key in year_keys:
            vi_path  = vi_files[date_key]
            doy_path = doy_files.get(date_key)
            date_obj = datetime.strptime(date_key, '%Y%m%d')

            vi = rxr.open_rasterio(vi_path, masked=True)
            
            # ── clip 1: HLS MGRS tile bounding box ───────────────────────────
            bbox_reproj = None
            if hls_bbox_gdf is not None:
                bbox_reproj = hls_bbox_gdf.to_crs(vi.rio.crs)
                vi = vi.rio.clip(bbox_reproj.geometry, bbox_reproj.crs,
                                 drop=True, from_disk=True)

            
            vi = vi.squeeze("band", drop=True)
            vi.attrs.pop("scale_factor", None)
            vi.attrs.pop("add_offset",   None)

            # ── clip 2: optional ROI ───────────────────────────
            roi_reproj = None
            if roi is not None:
                roi_reproj = roi.to_crs(vi.rio.crs)
                vi = vi.rio.clip(roi_reproj.geometry, roi_reproj.crs, drop=True)

                
            if doy_path:
                doy_da = rxr.open_rasterio(doy_path, masked=True).squeeze("band", drop=True)
                doy_da.attrs.pop("scale_factor", None)
                doy_da.attrs.pop("add_offset",   None)

                if roi_reproj is not None:
                    doy_da = doy_da.rio.clip(roi_reproj.geometry, roi_reproj.crs, drop=True)

                doy_da   = doy_da.where(vi.notnull() & (doy_da >= 1) & (doy_da <= 366))
                mean_doy = float(doy_da.mean(skipna=True).values)

                if np.isnan(mean_doy):
                    print(f"Warning: No valid DOY pixels for {date_key}, using composite date")
                    rep_date = date_obj
                else:
                    rep_date = datetime(date_obj.year, 1, 1) + timedelta(days=round(mean_doy) - 1)
            else:
                print(f"Warning: No DOY tif for {date_key}, using composite date")
                mean_doy = None
                rep_date = date_obj

            valid = vi.values[~np.isnan(vi.values)]
            if valid.size == 0:
                print(f"Warning: No valid VI pixels for {date_key} — skipping")
                del vi
                gc.collect()
                continue

            try:
                all_composites_modis.append(vi.assign_coords(time=rep_date).expand_dims("time"))
            except UnboundLocalError:
                pass

            if roi is None:
                AOI = "Tile"
            else:
                AOI = roi_name

            records.append({
                "date_key":            date_key,
                "composite_date":      date_obj,
                "composite_mean_date": rep_date,
                "mean_doy":            round(mean_doy, 2) if mean_doy is not None else None,
                f"{veg_index}_mean":   float(np.mean(valid)),
                f"{veg_index}_median": float(np.median(valid)),
                f"{veg_index}_min":    float(np.min(valid)),
                f"{veg_index}_max":    float(np.max(valid)),
                f"{veg_index}_stdev":  float(np.std(valid, ddof=1)),
                "n_valid_pixels":      int(valid.size),
            })

            annual_vi_stack.append(vi.assign_coords(time=rep_date).expand_dims("time"))

        if not annual_vi_stack:
            print(f"No valid composites for {year} — skipping annual tif")
            continue

        stacked     = xr.concat(annual_vi_stack, dim="time")
        annual_mean = stacked.mean(dim="time", skipna=True)
        annual_std  = stacked.std( dim="time", skipna=True, ddof=1)

        annual_2band = xr.concat(
            [annual_mean.expand_dims("band"),
             annual_std.expand_dims("band")],
            dim="band"
        ).assign_coords(band=[1, 2])

        annual_2band.attrs = {
            "long_name":    [f"{veg_index.upper()}_annual_mean", f"{veg_index.upper()}_annual_stdev"],
            "year":         year,
            "n_composites": len(annual_vi_stack),
        }
        vi_crs = annual_vi_stack[0].rio.crs
        annual_2band = annual_2band.rio.write_crs(vi_crs)

        annual_fname = f"{tile_id}_{veg_index.upper()}_annual_{year}_{veg_index.upper()}.tif"
        annual_path  = os.path.join(output_dir, annual_fname)
        annual_2band.rio.to_raster(annual_path, compress="lzw")
        print(f"Annual written: {annual_fname}  |  n_composites={len(annual_vi_stack)}")

        del stacked, annual_mean, annual_std, annual_2band, annual_vi_stack
        try:
            del vi
        except UnboundLocalError:
            pass
        gc.collect()

        print(f"Year {year} complete  |  running record count: {len(records)}")

    df = pd.DataFrame(records).sort_values("composite_date").reset_index(drop=True)
    if roi_name:
        csv_path = os.path.join(outdir, f"MOD13Q1_{veg_index.upper()}_timeseries_stats-{roi_name}.csv")
    else:
        csv_path = os.path.join(outdir, f"MOD13Q1_{veg_index.upper()}_timeseries_stats.csv")
    df.to_csv(csv_path, index=False)
    print(f"Timeseries CSV: {csv_path}")
    return df

In [ ]:
modis = compute_modis_vi_stats(veg_index,
                               f'/shared/users/hls_bdec/tommy_code/modis_veg_indices/{MGRS_TILE}/modis_tiff/MOD13Q1/', 
                               outdir = output_dir,
                               tile_id = MGRS_TILE,
                               start_year = int(START_DATE[:4]), 
                               end_year = int(END_DATE[:4]) + 1,
                               hls_data_dir=f'/shared/users/hls_bdec/tommy_code/hls_veg_indices/{MGRS_TILE}/',
                               roi_name = "cartercountry",
                               roi_shp='/shared/users/hls_bdec/tommy_code/roi_shp/cartercountry.zip')

In [ ]:
hls = compute_hls_vi_stats(veg_index, 
                           f'/shared/users/hls_bdec/tommy_code/hls_veg_indices/{MGRS_TILE}/', 
                           outdir = output_dir,              
                           tile_id = MGRS_TILE,
                           start_year = int(START_DATE[:4]), 
                           end_year = int(END_DATE[:4]) + 1,
                           roi_name = "cartercountry",
                           roi_shp='/shared/users/hls_bdec/tommy_code/roi_shp/cartercountry.zip')

## Spatial Stats

In [ ]:
# Compute spatial statistics
stacked_all = xr.concat(all_composites_hls, dim="time")
# stacked_all = xr.concat(all_composites_modis, dim="time")

In [ ]:
vi_mean = stacked_all.mean("time", skipna=True)
spatial_mean_mean = float(vi_mean.mean(skipna=True))
spatial_min_mean = float(vi_mean.min(skipna=True))
spatial_max_mean = float(vi_mean.max(skipna=True))

print(f"Spatial mean mean: {spatial_mean_mean:.3f}")
print(f"Spatial min mean: {spatial_min_mean:.3f}")
print(f"Spatial max mean: {spatial_max_mean:.3f}")

vi_min = stacked_all.min("time", skipna=True)
spatial_mean_min = float(vi_min.mean(skipna=True))
spatial_min_min = float(vi_min.min(skipna=True))
spatial_max_min = float(vi_min.max(skipna=True))

print(f"Spatial mean min: {spatial_mean_min:.3f}")
print(f"Spatial min min: {spatial_min_min:.3f}")
print(f"Spatial max min: {spatial_max_min:.3f}")

vi_max = stacked_all.max("time", skipna=True)
spatial_mean_max = float(vi_max.mean(skipna=True))
spatial_min_max = float(vi_max.min(skipna=True))
spatial_max_max = float(vi_max.max(skipna=True))

print(f"Spatial mean max: {spatial_mean_max:.3f}")
print(f"Spatial min max: {spatial_min_max:.3f}")
print(f"Spatial max max: {spatial_max_max:.3f}")

vi_dmax = stacked_all.idxmax("time", skipna=True)
vi_dmax = vi_dmax.dt.dayofyear
spatial_mean_dmax = float(vi_dmax.mean(skipna=True))
spatial_min_dmax = float(vi_dmax.min(skipna=True))
spatial_max_dmax = float(vi_dmax.max(skipna=True))

print(f"Spatial mean dmax: {spatial_mean_dmax:.3f}")
print(f"Spatial min dmax: {spatial_min_dmax:.3f}")
print(f"Spatial max dmax: {spatial_max_dmax:.3f}")

sos_list = []
eos_list = []
years = sorted(set(stacked_all["time"].dt.year.values))

for y in years:
    annual = stacked_all.sel(time=stacked_all["time"].dt.year == y)
    annual = annual.sortby("time")

    annual_min = annual.min("time")
    annual_max = annual.max("time")

    amplitude = annual_max - annual_min
    # Try to mask out any non-phenological pixels
    valid = amplitude > 0.1
    annual = annual.where(valid)

    threshold  = annual_min + 0.15 * amplitude
    above = annual >= threshold

    # SOS = first time above threshold
    first_idx = above.argmax("time")
    first_idx = first_idx.compute()
    sos = annual.time.isel(time=first_idx)

    # EOS = first time below threshold after peak
    peak = annual.idxmax("time")
    peak = peak.compute()
    after_peak = annual.time > peak
    below = annual < threshold
    end = after_peak & below
    # Make sure there is a dip after the peak
    has_end = end.any("time")
    first_end_idx = end.argmax("time")
    first_end_idx = first_end_idx.compute()
    eos = annual.time.isel(time=first_end_idx).where(has_end)

    sos_list.append(sos)
    eos_list.append(eos)

vi_sos = xr.concat(sos_list, dim="year").dt.dayofyear
spatial_mean_sos = float(vi_sos.mean(skipna=True))
spatial_min_sos = float(vi_sos.min(skipna=True))
spatial_max_sos = float(vi_sos.max(skipna=True))

print(f"Spatial mean sos: {spatial_mean_sos:.3f}")
print(f"Spatial min sos: {spatial_min_sos:.3f}")
print(f"Spatial max sos: {spatial_max_sos:.3f}")

vi_eos = xr.concat(eos_list, dim="year").dt.dayofyear
spatial_mean_eos = float(vi_eos.mean(skipna=True))
spatial_min_eos = float(vi_eos.min(skipna=True))
spatial_max_eos = float(vi_eos.max(skipna=True))

print(f"Spatial mean eos: {spatial_mean_eos:.3f}")
print(f"Spatial min eos: {spatial_min_eos:.3f}")
print(f"Spatial max eos: {spatial_max_eos:.3f}")

vi_seasonlen = vi_eos - vi_sos
spatial_mean_seasonlen = float(vi_seasonlen.mean(skipna=True))
spatial_min_seasonlen = float(vi_seasonlen.min(skipna=True))
spatial_max_seasonlen = float(vi_seasonlen.max(skipna=True))

print(f"Spatial mean seasonlen: {spatial_mean_seasonlen:.3f}")
print(f"Spatial min seasonlen: {spatial_min_seasonlen:.3f}")
print(f"Spatial max seasonlen: {spatial_max_seasonlen:.3f}")

## Temporal Stats

In [ ]:
def plot_vi_timeseries(
    hls_df,
    modis_df,
    veg_index,
    outdir,
    tile_id="HLS",
    stat="mean",
    plot_std=True,
    figsize=(14, 5),
    vi_min=None#0.6,        # ← new: set to None to disable filtering
):
    vi_col   = f"{veg_index.lower()}_{stat}"
    std_col  = f"{veg_index.lower()}_stdev"
    vi_upper = veg_index.upper()

    fig, ax = plt.subplots(figsize=figsize)

    # ── HLS timeseries ────────────────────────────────────────────────────────
    if not hls_df.empty and vi_col in hls_df.columns:
        hls_plot_df = hls_df.copy()
        hls_plot_df["composite_mean_date"] = pd.to_datetime(hls_plot_df["composite_mean_date"])
        hls_plot_df = hls_plot_df.sort_values("composite_mean_date").reset_index(drop=True)

        # ── filter below vi_min ───────────────────────────────────────────────
        if vi_min is not None:
            n_before = len(hls_plot_df)
            hls_plot_df = hls_plot_df[hls_plot_df[vi_col] >= vi_min].reset_index(drop=True)
            print(f"HLS: removed {n_before - len(hls_plot_df)} points below {vi_min} ({len(hls_plot_df)} remaining)")

        hls_x = hls_plot_df["composite_mean_date"]
        hls_y = hls_plot_df[vi_col]

        ax.plot(hls_x, hls_y,
                color="steelblue", linewidth=1.5, marker="o", markersize=3,
                label=f"HLS {vi_upper} ({stat})")

        if plot_std and std_col in hls_plot_df.columns:
            ax.fill_between(hls_x,
                            hls_y - hls_plot_df[std_col],
                            hls_y + hls_plot_df[std_col],
                            color="steelblue", alpha=0.15, label="HLS ±1σ")
    else:
        print(f"Warning: HLS DataFrame is empty or missing '{vi_col}' — skipping HLS plot")
        print(f"  Available columns: {list(hls_df.columns)}")

    # ── MODIS timeseries ──────────────────────────────────────────────────────
    if not modis_df.empty and vi_col in modis_df.columns:
        modis_plot_df = modis_df.copy()
        modis_plot_df["composite_mean_date"] = pd.to_datetime(modis_plot_df["composite_mean_date"])
        modis_plot_df = modis_plot_df.sort_values("composite_mean_date").reset_index(drop=True)

        # ── filter below vi_min ───────────────────────────────────────────────
        if vi_min is not None:
            n_before = len(modis_plot_df)
            modis_plot_df = modis_plot_df[modis_plot_df[vi_col] >= vi_min].reset_index(drop=True)
            print(f"MODIS: removed {n_before - len(modis_plot_df)} points below {vi_min} ({len(modis_plot_df)} remaining)")

        mod_x = modis_plot_df["composite_mean_date"]
        mod_y = modis_plot_df[vi_col]

        ax.plot(mod_x, mod_y,
                color="darkorange", linewidth=1.5, marker="s", markersize=3,
                label=f"MODIS {vi_upper} ({stat})")

        if plot_std and std_col in modis_plot_df.columns:
            ax.fill_between(mod_x,
                            mod_y - modis_plot_df[std_col],
                            mod_y + modis_plot_df[std_col],
                            color="darkorange", alpha=0.15, label="MODIS ±1σ")
    else:
        print(f"Warning: MODIS DataFrame is empty or missing '{vi_col}' — skipping MODIS plot")
        print(f"  Available columns: {list(modis_df.columns)}")

    # ── Formatting ────────────────────────────────────────────────────────────
    ax.set_title(f"{tile_id}  |  {vi_upper} Timeseries — HLS vs MODIS", fontsize=13)
    ax.set_xlabel("Date")
    ax.set_ylabel(vi_upper)
    ax.legend(loc="best", fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    fig.tight_layout()

    png_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_HLS_vs_MODIS_timeseries.png")
    fig.savefig(png_path, dpi=150)
    plt.show()
    print(f"Plot saved: {png_path}")
    return fig, ax

In [ ]:
plot_vi_timeseries(hls, modis, veg_index, output_dir, tile_id=MGRS_TILE)

In [ ]:
def plot_vi_seasonal_mean(
    hls_df,
    modis_df,
    veg_index,
    outdir,
    tile_id="HLS",
    stat="mean",
    vi_min=None,
    hls_bin_width=10,      # matches HLS 10-day composite cadence
    modis_bin_width=16,    # matches MODIS 16-day composite cadence
    figsize=(13, 5),
):
    """
    Seasonal plot showing the multi-year mean ± 1 std for each sensor
    on a single set of axes. Each sensor uses its native composite
    cadence as the bin width so DOY bins are not artificially widened.

    Parameters
    ----------
    hls_df          : DataFrame from compute_hls_vi_stats()
    modis_df        : DataFrame from compute_modis_vi_stats()
    veg_index       : str    e.g. 'NDVI'
    outdir          : str    directory to save PNG
    tile_id         : str    used in title / filename
    stat            : str    'mean' or 'median'
    vi_min          : float | None   drop composites below this value
    hls_bin_width   : int    DOY bin width for HLS   (default 10)
    modis_bin_width : int    DOY bin width for MODIS (default 16)
    figsize         : tuple
    """

    vi_col   = f"{veg_index.lower()}_{stat}"
    vi_upper = veg_index.upper()

    # ── helper: prep + bin ────────────────────────────────────────────────────
    def _prep(df, date_col, bin_width):
        d = df.copy()
        d[date_col] = pd.to_datetime(d[date_col])
        d = d.sort_values(date_col).reset_index(drop=True)
        d["_year"] = d[date_col].dt.year
        d["_doy"]  = d[date_col].dt.dayofyear
        if vi_min is not None:
            d = d[d[vi_col] >= vi_min].reset_index(drop=True)

        # Bin to sensor-native cadence
        d["_doy_bin"] = (((d["_doy"] - 1) // bin_width) * bin_width
                         + bin_width // 2 + 1)
        return d

    # ── helper: cross-year mean & std per DOY bin ─────────────────────────────
    def _seasonal_stats(df):
        return (
            df.groupby("_doy_bin")[vi_col]
            .agg(bin_mean="mean", bin_std="std", n_years="count")
            .reset_index()
            .sort_values("_doy_bin")
        )

    # ── per-sensor config — note separate bin_width per sensor ────────────────
    sensor_cfg = {
        "HLS":   dict(df=hls_df,   date_col="composite_mean_date",
                      color="steelblue",  marker="o",
                      bin_width=hls_bin_width),
        "MODIS": dict(df=modis_df, date_col="composite_mean_date",
                      color="darkorange", marker="s",
                      bin_width=modis_bin_width),
    }

    fig, ax = plt.subplots(figsize=figsize)

    for sensor, cfg in sensor_cfg.items():
        df_in = cfg["df"]
        if df_in.empty or vi_col not in df_in.columns:
            print(f"Warning: {sensor} DataFrame empty or missing '{vi_col}' — skipping")
            continue

        prepped = _prep(df_in, cfg["date_col"], cfg["bin_width"])
        stats   = _seasonal_stats(prepped)

        x     = stats["_doy_bin"].values
        ymean = stats["bin_mean"].values
        ystd  = stats["bin_std"].fillna(0).values    # single-year bins → std=NaN → 0

        # ── mean line ─────────────────────────────────────────────────────────
        ax.plot(x, ymean,
                color=cfg["color"], linewidth=2.0,
                marker=cfg["marker"], markersize=4,
                label=f"{sensor} {vi_upper} mean  (bin={cfg['bin_width']}d)",
                zorder=3)

        # ── ±1 std shading ────────────────────────────────────────────────────
        ax.fill_between(x,
                        ymean - ystd,
                        ymean + ystd,
                        color=cfg["color"], alpha=0.18,
                        label=f"{sensor} ±1σ",
                        zorder=2)

    # ── month x-tick labels ───────────────────────────────────────────────────
    month_doys   = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
    month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"]
    ax.set_xticks(month_doys)
    ax.set_xticklabels(month_labels)
    ax.set_xlim(1, 365)

    ax.set_title(f"{tile_id}  |  {vi_upper} Multi-Year Seasonal Mean ± 1σ  "
                 f"(HLS {hls_bin_width}d  |  MODIS {modis_bin_width}d)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel(vi_upper)
    ax.legend(fontsize=9, loc="best")
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()

    png_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_seasonal_mean_std.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {png_path}")
    return fig, ax

In [ ]:
plot_vi_seasonal_mean(hls, modis, veg_index, output_dir, tile_id=MGRS_TILE,
                      vi_min=None)

In [ ]:
def compute_annual_phenology_summary(hls_df, modis_df, veg_index, outdir, tile_id="HLS",
                                     vi_min_filter=None):
    """
    For each year and sensor compute:
      - Annual min / max VI
      - Start of Season (SOS): first date where VI >= min + 15% * (max - min)
      - End of Season (EOS): last  date where VI >= min + 15% * (max - min)
      - VI value and DOY at SOS / EOS

    The 15% amplitude threshold is a standard phenology approach:
        threshold = annual_min + 0.15 * (annual_max - annual_min)

    Parameters
    ----------
    hls_df        : DataFrame from compute_hls_vi_stats()
    modis_df      : DataFrame from compute_modis_vi_stats()
    veg_index     : str   e.g. 'NDVI'
    outdir        : str   directory to save CSV and PNG
    tile_id       : str   used in titles / filenames
    vi_min_filter : float | None   drop composites below this value before
                    phenology calculation (same filter as the timeseries plot)
    """

    vi_col   = f"{veg_index.lower()}_mean"
    vi_upper = veg_index.upper()

    # ── helpers ───────────────────────────────────────────────────────────────
    def _prep(df, date_col):
        """Return a clean copy sorted by date with year and DOY columns."""
        d = df.copy()
        d[date_col] = pd.to_datetime(d[date_col])
        d = d.sort_values(date_col).reset_index(drop=True)
        d["_year"] = d[date_col].dt.year
        d["_doy"]  = d[date_col].dt.dayofyear
        if vi_min_filter is not None:
            d = d[d[vi_col] >= vi_min_filter].reset_index(drop=True)
        return d

    def _annual_stats(df, sensor_label):
        """Compute per-year phenology stats for one sensor DataFrame."""
        rows = []
        for year, grp in df.groupby("_year"):
            grp = grp.sort_values("_doy").reset_index(drop=True)
            vi  = grp[vi_col].values
            doy = grp["_doy"].values

            ann_min = float(np.nanmin(vi))
            ann_max = float(np.nanmax(vi))
            amp     = ann_max - ann_min

            # 15 % amplitude threshold
            # interpolate | spline == continuous time series
            threshold = ann_min + 0.15 * amp

            above = grp[grp[vi_col] >= threshold]
            
            if above.empty:
                sos_doy = sos_vi = eos_doy = eos_vi = np.nan
            else:
                sos_row = above.iloc[0]
                eos_row = above.iloc[-1]
                sos_doy = int(sos_row["_doy"])
                sos_vi  = float(sos_row[vi_col])
                eos_doy = int(eos_row["_doy"])
                eos_vi  = float(eos_row[vi_col])

            rows.append({
                "sensor":       sensor_label,
                "year":         int(year),
                "annual_min":   round(ann_min, 4),
                "annual_max":   round(ann_max, 4),
                "amplitude":    round(amp, 4),
                "threshold_15pct": round(threshold, 4),
                "sos_doy":      sos_doy,
                "sos_vi":       round(sos_vi,  4) if not np.isnan(sos_vi)  else np.nan,
                "eos_doy":      eos_doy,
                "eos_vi":       round(eos_vi,  4) if not np.isnan(eos_vi)  else np.nan,
                "season_length_days": (eos_doy - sos_doy) if not np.isnan(sos_doy) else np.nan,
            })
        return pd.DataFrame(rows)

    # ── compute for each sensor ───────────────────────────────────────────────
    dfs = []
    if not hls_df.empty and vi_col in hls_df.columns:
        hls_prepped = _prep(hls_df,   "composite_mean_date")
        dfs.append(_annual_stats(hls_prepped, "HLS"))
    else:
        print(f"Warning: HLS DataFrame empty or missing '{vi_col}'")

    if not modis_df.empty and vi_col in modis_df.columns:
        mod_prepped = _prep(modis_df, "composite_mean_date")
        dfs.append(_annual_stats(mod_prepped, "MODIS"))
    else:
        print(f"Warning: MODIS DataFrame empty or missing '{vi_col}'")

    if not dfs:
        print("No data to summarise.")
        return pd.DataFrame()

    summary = pd.concat(dfs, ignore_index=True).sort_values(["year", "sensor"])

    csv_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_annual_phenology_summary.csv")
    summary.to_csv(csv_path, index=False)
    print(f"Summary CSV saved: {csv_path}")
    print(summary.to_string(index=False))

    # ── plotting ──────────────────────────────────────────────────────────────
    #_plot_phenology_summary(summary, veg_index, outdir, tile_id)
    _plot_phenology(summary, veg_index, outdir, tile_id)
    
    return summary


# ─────────────────────────────────────────────────────────────────────────────
def _plot_phenology_summary(summary_df, veg_index, outdir, tile_id):
    """
    Point plot: year on x-axis, one panel per metric,
    HLS = steelblue circles, MODIS = darkorange squares.
    """

    vi_upper = veg_index.upper()

    metrics = [
        ("annual_min",   f"{vi_upper} Annual Min",     "VI"),
        ("annual_max",   f"{vi_upper} Annual Max",     "VI"),
        ("sos_doy",      "Start of Season (SOS)",       "DOY"),
        ("sos_vi",       f"{vi_upper} at SOS",          "VI"),
        ("eos_doy",      "End of Season (EOS)",         "DOY"),
        ("eos_vi",       f"{vi_upper} at EOS",          "VI"),
        ("season_length_days", "Season Length",         "Days"),
    ]

    n_panels = len(metrics)
    ncols    = 2
    nrows    = math.ceil(n_panels / ncols)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(13, 4 * nrows),
                             sharex=False)
    axes = axes.flatten()

    sensor_style = {
        "HLS":   dict(color="steelblue",  marker="o", zorder=3),
        "MODIS": dict(color="darkorange", marker="s", zorder=3),
    }

    for ax, (col, title, ylabel) in zip(axes, metrics):
        for sensor, style in sensor_style.items():
            sub = summary_df[summary_df["sensor"] == sensor].dropna(subset=[col])
            if sub.empty:
                continue
            ax.scatter(sub["year"], sub[col],
                       label=sensor, s=60,
                       **style)
            # connect points with a thin line to help readability
            ax.plot(sub["year"], sub[col],
                    color=style["color"], linewidth=0.8,
                    alpha=0.5, zorder=2)

        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Year")
        ax.set_ylabel(ylabel)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.legend(fontsize=8, loc="best")
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

    # hide any unused panel (if n_panels is odd)
    for ax in axes[n_panels:]:
        ax.set_visible(False)

    fig.suptitle(f"{tile_id}  |  {vi_upper} Annual Phenology Summary",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()

    png_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_annual_phenology_summary.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {png_path}")

In [ ]:
def _plot_phenology(summary_df, veg_index, outdir, tile_id):

    vi_upper = veg_index.upper()

    metrics = [
        ("annual_min", "Annual Min", "VI"),
        ("annual_max", "Annual Max", "VI"),
        ("amplitude", f"{vi_upper} Range", "VI"),
        ("sos_doy", "Start of Season (SOS)", "DOY"),
        ("sos_vi", f"{vi_upper} at SOS", "VI"),
        ("eos_doy", "End of Season (EOS)", "DOY"),
        ("eos_vi", f"{vi_upper} at EOS", "VI"),
        ("season_length_days", "Season Length", "Days"),
    ]

    sensor_style = {
        "HLS": dict(color="steelblue", marker="o", zorder=3),
        "MODIS": dict(color="darkorange", marker="s", zorder=3),
    }

    for col, title, ylabel in metrics:
        fig, ax = plt.subplots(figsize=(8, 5))
        for sensor, style in sensor_style.items():
            sub = summary_df[summary_df["sensor"] == sensor].dropna(subset=[col])
            if sub.empty:
                continue
            ax.scatter(sub["year"], sub[col],
                       label=sensor, s=60,
                       **style)
            ax.plot(sub["year"], sub[col],
                    color=style["color"], linewidth=0.8,
                    alpha=0.5,zorder=2)

        ax.set_title(f"{tile_id} | {title}", fontsize=12)
        ax.set_xlabel("Year")
        ax.set_ylabel(ylabel)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.legend(fontsize=9)
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
        fig.tight_layout()

        png_path = os.path.join(
            outdir,
            f"{tile_id}_{vi_upper}_{col}.png"
        )

        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Plot saved: {png_path}")

In [ ]:
summary_df = compute_annual_phenology_summary(
    hls, 
    modis, 
    veg_index, 
    output_dir,
    tile_id=MGRS_TILE,
    vi_min_filter=None#0.6,    
)

In [ ]:
def compute_annual_phenology_summary(hls_df, modis_df, veg_index, outdir, tile_id="HLS",
                                     vi_min_filter=None):
    """
    For each year and sensor compute:
      - Annual min VI + its DOY
      - Annual max VI + its DOY
    """

    vi_col   = f"{veg_index.lower()}_mean"
    vi_upper = veg_index.upper()

    # ── helpers ───────────────────────────────────────────────────────────────
    def _prep(df, date_col):
        d = df.copy()
        d[date_col] = pd.to_datetime(d[date_col])
        d = d.sort_values(date_col).reset_index(drop=True)
        d["_year"] = d[date_col].dt.year
        d["_doy"]  = d[date_col].dt.dayofyear
        if vi_min_filter is not None:
            d = d[d[vi_col] >= vi_min_filter].reset_index(drop=True)
        return d

    def _annual_stats(df, sensor_label):
        rows = []
        for year, grp in df.groupby("_year"):
            grp = grp.sort_values("_doy").reset_index(drop=True)

            # ── min ───────────────────────────────────────────────────────────
            min_idx = grp[vi_col].idxmin()
            ann_min = float(grp.loc[min_idx, vi_col])
            min_doy = int(grp.loc[min_idx, "_doy"])

            # ── max ───────────────────────────────────────────────────────────
            max_idx = grp[vi_col].idxmax()
            ann_max = float(grp.loc[max_idx, vi_col])
            max_doy = int(grp.loc[max_idx, "_doy"])

            rows.append({
                "sensor":    sensor_label,
                "year":      int(year),
                "annual_min": round(ann_min, 4),
                "min_doy":    min_doy,
                "annual_max": round(ann_max, 4),
                "max_doy":    max_doy,
            })
        return pd.DataFrame(rows)

    # ── compute for each sensor ───────────────────────────────────────────────
    dfs = []
    if not hls_df.empty and vi_col in hls_df.columns:
        dfs.append(_annual_stats(_prep(hls_df,   "composite_mean_date"), "HLS"))
    else:
        print(f"Warning: HLS DataFrame empty or missing '{vi_col}'")

    if not modis_df.empty and vi_col in modis_df.columns:
        dfs.append(_annual_stats(_prep(modis_df, "composite_mean_date"), "MODIS"))
    else:
        print(f"Warning: MODIS DataFrame empty or missing '{vi_col}'")

    if not dfs:
        print("No data to summarise.")
        return pd.DataFrame()

    summary = pd.concat(dfs, ignore_index=True).sort_values(["year", "sensor"])

    csv_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_annual_phenology_summary.csv")
    summary.to_csv(csv_path, index=False)
    print(f"Summary CSV saved: {csv_path}")
    print(summary.to_string(index=False))

    _plot_phenology_summary(summary, veg_index, outdir, tile_id)

    return summary


# ─────────────────────────────────────────────────────────────────────────────
def _plot_phenology_summary(summary_df, veg_index, outdir, tile_id):
    """
    4-panel point plot:
      Top row    : Annual Min VI  |  Annual Max VI
      Bottom row : DOY of Min     |  DOY of Max
    HLS = steelblue circles, MODIS = darkorange squares.
    """

    vi_upper = veg_index.upper()

    # (column, title, y-axis label)
    metrics = [
        ("annual_min", f"{vi_upper} Annual Min",  "VI"),
        ("annual_max", f"{vi_upper} Annual Max",  "VI"),
        ("min_doy",    "DOY of Annual Min",        "DOY"),
        ("max_doy",    "DOY of Annual Max",        "DOY"),
    ]

    sensor_style = {
        "HLS":   dict(color="steelblue",  marker="o"),
        "MODIS": dict(color="darkorange", marker="s"),
    }

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    axes = axes.flatten()

    for ax, (col, title, ylabel) in zip(axes, metrics):
        for sensor, style in sensor_style.items():
            sub = summary_df[summary_df["sensor"] == sensor].dropna(subset=[col])
            if sub.empty:
                continue
            ax.scatter(sub["year"], sub[col],
                       label=sensor, s=60, zorder=3, **style)
            ax.plot(sub["year"], sub[col],
                    color=style["color"], linewidth=0.8, alpha=0.5, zorder=2)

        ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.legend(fontsize=8, loc="best")
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

    # only bottom row needs x label
    for ax in axes[2:]:
        ax.set_xlabel("Year")

    plt.setp([ax.get_xticklabels() for ax in axes[:2]], visible=False)

    fig.suptitle(f"{tile_id}  |  {vi_upper} Annual Min / Max  &  DOY",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()

    png_path = os.path.join(outdir, f"{tile_id}_{vi_upper}_annual_phenology_summary.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {png_path}")

In [ ]:
hls.head()

In [ ]:
modis

### Plot mean NDVI, EVI, and EVI2 and % pixel coverage over time
- Tells us how NDVI, EVI, and EVI2 change over time
- and how complete the pixel coverage is

In [ ]:
# amount of non-NaN (not a number) pixels over the total number of pixels in the ROI
valid_pixel_pct = (tenday_ndvis.count(dim = ['x','y']) / (tenday_ndvis[0].shape[0] * tenday_ndvis[0].shape[1]))
# valid_pixel_pct

In [ ]:
# take ROI mean NDVI, EVI, EVI2 stats
roi_mean_ndvi = tenday_ndvis.mean(['x', 'y'], skipna=True)
roi_mean_evi = tenday_evis.mean(['x', 'y'], skipna=True)
roi_mean_evi2 = tenday_evi2s.mean(['x', 'y'], skipna=True)

plt.figure(figsize=(10, 6))
plt.plot(roi_mean_ndvi.time, roi_mean_ndvi, label = 'NDVI')
plt.plot(roi_mean_evi.time, roi_mean_evi, label = 'EVI')
plt.plot(roi_mean_evi2.time, roi_mean_evi2, label = 'EVI2')
plt.plot(roi_mean_ndvi.time, valid_pixel_pct, label = "% pixel coverage", alpha = 0.5, linewidth = 0.5)
# plt.xlim(pd.Timestamp('2018-01-01'), pd.Timestamp('2018-12-31'))

zero_indices = np.where(valid_pixel_pct.values == 0)[0]
zero_x_coords = valid_pixel_pct.coords['time'].values[zero_indices]

for x_val in zero_x_coords:
    plt.axvline(x_val, ymax=1, color='gray', linestyle='--', linewidth=0.2)

plt.title(f"NDVI, EVI, EVI2 for SERC {tile_id} monthly composite and pixel coverage")
plt.xlabel('')
plt.ylabel('NDVI, EVI, EVI2')
plt.legend()
plt.show()

There are some spaces in the NDVI, EVI, and EVI2 data. These appear to be highly correlated with % pixel coverage, meaning that when there is very low pixel coverage (high numbers of NaN pixels), there is not enough data to construct the vegetation indices. These periods of low valid pixels could be very cloudy times of the year, or the HLS data could have been contaminated.

This graph shows a time in 2022 when EVI, EVI2, and NDVI all fall. We will look more later to see if this is caused by climatic factors or an error in composite creation. 

### Spatial standard deviation map

Calculates how NDVI changes over time for each pixel over the entire 10-year range. Pixels where NDVI drastically changes over time are worth looking deeper into, as this could indicate contamination of data. 

In [ ]:
# spatial standard deviation map 

# compute std dev across time dimension
evi_std_map = tenday_evis.std(dim="time", skipna=True)

# flag suspicious pixels (high temporal variance = likely contamination)
threshold = 0.1 
suspicious = evi_std_map > threshold

# plot
fig, ax = plt.subplots(figsize=(6, 6))

# full year std dev
evi_std_map.plot(
    cmap="YlOrRd",
    cbar_kwargs={"label": "Std Dev (EVI)"})
ax.set_title("Temporal Standard Deviation")

# overlay ROI boundary
roi_projected = roi_shp.to_crs(tenday_evis.rio.crs)
roi_projected.boundary.plot(ax=ax, color="black", linewidth=1)

plt.show()

# summary stats on the std map
print(f"Mean std across ROI: {float(evi_std_map.mean()):.4f}")
print(f"Max std across ROI: {float(evi_std_map.max()):.4f}")
print(f"% pixels above threshold: {float(suspicious.mean()) * 100:.1f}%")

### Visualize Seasonal EVI, EVI2, and NDVI Curves
This tile is in Madagascar, which is located in the Southern Hemisphere and therefore experiences summer from  around October-April. During this time, the region we are focusing on (in the Eastern portion of the country) is very humid with high rainfall. 

Run the code below to visualize how this effects EVI, EVI2, and NDVI.

In [ ]:
# take ROI mean NDVI, EVI, EVI2 stats
roi_mean_ndvi = tenday_ndvis.mean(['x', 'y'], skipna=True)
roi_mean_evi = tenday_evis.mean(['x', 'y'], skipna=True)
roi_mean_evi2 = tenday_evi2s.mean(['x', 'y'], skipna=True)

def shift_doy(doy, start_month_doy=152):
    # 152 = typical year June 1 day of year
    return (doy - start_month_doy) % 365

years = sorted(set(roi_mean_ndvi.time.dt.year.values))
n_years = len(years)
colors = cm.viridis(np.linspace(0, 1, n_years))

month_labels = ["Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May"]
month_doys = [1, 30, 61, 91, 122, 152, 182, 213, 244, 274, 305, 335]

index_arrays = {
    "ndvi": roi_mean_ndvi,
    "evi": roi_mean_evi,
    "evi2": roi_mean_evi2
}

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharey=False)

for ax, idx in zip(axes, ["ndvi", "evi", "evi2"]):
    roi_mean = index_arrays[idx]
    for i, (year, year_data) in enumerate(roi_mean.groupby("time.year")):
        shifted_doy = shift_doy(year_data.time.dt.dayofyear.values)
        sort_idx = np.argsort(shifted_doy)
        ax.plot(shifted_doy[sort_idx], year_data.values[sort_idx],
                color=colors[i], label=str(year), alpha=0.8, linewidth=1.5)
    
    ax.set_xticks(month_doys)
    ax.set_xticklabels(month_labels)
    ax.set_title(f"{idx.upper()} Seasonal Cycle (Jun–May)")
    ax.set_xlabel("Month")
    ax.set_ylabel(idx.upper())
    ax.grid(True, alpha=0.3)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharey=False)

for ax, idx in zip(axes, ["ndvi", "evi", "evi2"]):
    roi_mean = index_arrays[idx]
    for i, (year, year_data) in enumerate(roi_mean.groupby("time.year")):
        shifted_doy = shift_doy(year_data.time.dt.dayofyear.values)
        sort_idx = np.argsort(shifted_doy)
        ax.plot(shifted_doy[sort_idx], year_data.values[sort_idx],
                color=colors[i], label=str(year), alpha=0.8, linewidth=1.5)
    
    ax.set_xticks(month_doys)
    ax.set_xticklabels(month_labels)
    ax.set_title(f"{idx.upper()} Seasonal Cycle (Jun–May)")
    ax.set_xlabel("Month")
    ax.set_ylabel(idx.upper())
    ax.set_xlim(240, 300)
    ax.grid(True, alpha=0.3)

axes[0].set_ylim(0.6, 0.9)
axes[1].set_ylim(0.4, 0.7)
axes[2].set_ylim(0.4, 0.7)
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

EVI and EVI2 both experience peaks in the summer months. This makes sense as warmer temperatures and increased sunlight drive photosynthesis, leaf development, and canopy density. In summer, more lush vegetation increases the absorption of blue light and scattering of NIR light, directly resulting in a higher EVI signal.

Since NDVI is less sensitive in areas of high vegetation, as in this region of Madagascar where it is vegetated year-round, it does not change much over the course of a year. 

## Remove outlier composites

As you can see from the visualization above, there are some HLS composites that have much lower EVI/EVI2/NDVI values than is normal. While these inconsistencies might be caused by climatic influences, it is also likely that they are caused by errors in compositing. 

The following code:
1. Picks a threshold
2. Calculates the median NDVI/EVI/EVI2 value for each 10-day composite
3. Calculates the baseline median EVI/EVI2/NDVI value for each month across all years
4. Determines composites for which the percent deviation of its median value from the baseline median value is greater than the threshold
5. Removes these composites from data_cleaned
6. Prints the spatial visualization of the composites -- this is where your human eye should determine if the large percent deviation is from compositing errors or potentially climatic data. If there are multiple composites that are not made up of mostly NaN pixels (appearing white), increase the threshold and run again. 

In [ ]:
threshold = 0.15  # 15% deviation from monthly median
data = tenday_evis #tenday_ndvis, tenday_evi2s

# compute median per timestep for entire roi 
# single value per composite representing the whole ROI
roi_median_evi = tenday_evis.median(["x", "y"], skipna=True) 

# compute monthly baseline median - for each month, what is the median roi across all years
monthly_baseline_evi = roi_median_evi.groupby("time.month").median() 

# flag outlier composites
# for each timestep, get its month's baseline value
baseline_per_timestep_evi = monthly_baseline_evi.sel(month=roi_median_evi.time.dt.month)

# compute percent deviation from monthly baseline
pct_deviation_evi = abs(roi_median_evi - baseline_per_timestep_evi) / baseline_per_timestep_evi

# flag composites where deviation exceeds threshold
outlier_flag_evi = pct_deviation_evi > threshold

print(f"Total composites: {len(outlier_flag_evi)}")
print(f"Flagged as outliers: {int(outlier_flag_evi.sum())}")
print(f"Flagged dates: {data.time.values[outlier_flag_evi.values]}")

# remove flagged composites 
data_cleaned_evi = data.isel(time=~outlier_flag_evi.values)

# visualize flagged composites 
flagged_times = data.time.values[outlier_flag_evi.values]
n_flagged = len(flagged_times)

if n_flagged > 0:
    ncols = 3
    nrows = int(np.ceil(n_flagged / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = axes.flatten()
    for i, t in enumerate(flagged_times):
        data.sel(time=t).plot(ax=axes[i], cmap="RdYlGn", vmin=0, vmax=1)
        axes[i].set_title(f"{str(t)[:10]}\ndev: {float(pct_deviation_evi.sel(time=t)):.1%}")
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")
    # hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Flagged Outlier Composites (>10% from monthly median)", size=14)
    plt.tight_layout()
    plt.show()
else:
    print("No outliers flagged - consider raising the threshold")

Once the code above has successfully identified mostly NaN composites, save `data_cleaned` to the EVI, EVI2, and NDVI mean values and proceed. 

In [ ]:
# take ROI mean NDVI, EVI, EVI2 stats
tenday_evis = data_cleaned_evi

roi_mean_ndvi = tenday_ndvis.mean(['x', 'y'], skipna=True)
roi_mean_evi = data_cleaned_evi.mean(['x', 'y'], skipna=True)
roi_mean_evi2 = tenday_evi2s.mean(['x', 'y'], skipna=True)

def shift_doy(doy, start_month_doy=152):
    # 152 = typical year June 1 day of year
    return (doy - start_month_doy) % 365

years = sorted(set(roi_mean_ndvi.time.dt.year.values))
n_years = len(years)
colors = cm.viridis(np.linspace(0, 1, n_years))

month_labels = ["Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May"]
month_doys = [1, 30, 61, 91, 122, 152, 182, 213, 244, 274, 305, 335]

index_arrays = {
    "ndvi": roi_mean_ndvi,
    "evi": roi_mean_evi,
    "evi2": roi_mean_evi2
}

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharey=False)

for ax, idx in zip(axes, ["ndvi", "evi", "evi2"]):
    roi_mean = index_arrays[idx]
    for i, (year, year_data) in enumerate(roi_mean.groupby("time.year")):
        shifted_doy = shift_doy(year_data.time.dt.dayofyear.values)
        sort_idx = np.argsort(shifted_doy)
        ax.plot(shifted_doy[sort_idx], year_data.values[sort_idx],
                color=colors[i], label=str(year), alpha=0.8, linewidth=1.5)
    
    ax.set_xticks(month_doys)
    ax.set_xticklabels(month_labels)
    ax.set_title(f"{idx.upper()} Seasonal Cycle (Jun–May)")
    ax.set_xlabel("Month")
    ax.set_ylabel(idx.upper())
    ax.grid(True, alpha=0.3)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Import the 'DPT array' from MERRA2 used for HLS
**What is MERRA2?**
- Modern-Era Retrospective analysis for Research and Applications, Version 2
- 30-year (1980-present) global climate reanalysis dataset
  - Reanalysis: When the same climate model is used throughout an entire data period to provide consistent reporting of climate information, even if data sources are different
- Connects aerosol impacts with meteorological, chemical, and physical processes.

**MERRA2 DPT array in HLS composite analysis**
- DPT (Date, Precipitation, Temperature) daily array for a specific site from 2016-2025 (temporal extent of HLS composites)
- precipitation is a daily sum of `PRECTOTCORR` - the bias-corrected total precipitation, in `cm`  
- temperature is a daily mean of `TLML` variable - the time-averaged (to hourly) surface (2m) air temperature, in `C`

In [ ]:
# Local filepath to MERRA2 data
merra2_dptarr_filepath_list = [fn for fn in glob.glob('/shared/users/sapolinsky/MERRA2_files/ignore_for_now/DPTarr_MERRA2_E*_S0*.npy') if 'bad_date_field' not in fn]

#### Parse the filename to get the centroid of latitude and longitude associated with the MERRA2 data

In [ ]:
list_lat = []
list_lon = []
filepath_list = []

index_to_insert_decimal = 4

for filepath in merra2_dptarr_filepath_list:
    for p in [i for i in filepath.strip('.npy').split('_')[-2:]]:
        s = re.sub('\D', '', p)
        s = s[:index_to_insert_decimal] + '.' + s[index_to_insert_decimal:]
        
        if 'W' in p: lon = float(s) * -1e-1
        if 'E' in p: lon = float(s) * 1e-1
        if 'N' in p: lat = float(s) * 1e-1
        if 'S' in p: lat = float(s) * -1e-1
        
    if lon >= -180. and lon <= 180:
        list_lat.append(lat)
        list_lon.append(lon)
        filepath_list.append(filepath)

In [ ]:
print(list_lat)
print(list_lon)
print(filepath_list)

### Map the general area of this MERRA2 DPT array over the HLS Madagascar sites
MERRA2 has a fixed grid resolution of 0.625° longitude × 0.5° latitude. We already have the midpoints of each MERRA2 tile, so construct the outlines and plot to visualize where the MERRA2 tiles overlap with the HLS tiles.

In [ ]:
# MERRA2 GeoDataFrame
merra2_gdf = gpd.GeoDataFrame(
    {'filename': filepath_list},
    geometry=gpd.points_from_xy(list_lon, list_lat),
    crs='EPSG:4326'
)

m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

folium.GeoJson(gdf_geo, 
               style_function=lambda x: {'color': 'red', 'weight': 2},
               name="SERC bounds"
              ).add_to(m)

folium.raster_layers.ImageOverlay(
    image=image,
    bounds=[[bounds[1], bounds[0]], [bounds[3], bounds[2]]],
    opacity=0.7,
    name="NDVI from 2020 June"
).add_to(m)

# using the lat and lon midpoint values of each DPT MERRA2 npy file, construct the outlines
merra2_boxes_gdf = gpd.GeoDataFrame(
    {'filename': filepath_list},
    geometry=[
        box(lon - 0.625/2, lat - 0.5/2, lon + 0.625/2, lat + 0.5/2)
        for lon, lat in zip(list_lon, list_lat)
    ],
    crs='EPSG:4326'
)

folium.GeoJson(merra2_boxes_gdf,
               style_function=lambda x: {'color': 'blue', 'weight': 2, 'fillOpacity': 0.1},
               tooltip=folium.GeoJsonTooltip(fields=['filename']),
               name="MERRA2 footprints"
              ).add_to(m)

folium.LayerControl().add_to(m)
m

### Load the DPT array file into a pandas dataframe

In [ ]:
df = pd.DataFrame({'filepath': filepath_list,'longitude': list_lon, 'latitude': list_lat})
df['filename'] = df['filepath'].apply(os.path.basename)
dpt_array_points_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(list_lon, list_lat), crs=4326).reset_index()

In [ ]:
dpt_array_df_list = []
for index, row in dpt_array_points_gdf.iterrows():
    filepath = row['filepath']
    data = np.load(filepath)
    
    # Creating pandas dataframe from numpy array
    dpt_array_df = pd.DataFrame({'filepath': filepath,'date': data[:, 0], 'precip': data[:, 1], 'temp': data[:, 2]})
    dpt_array_df['filename'] = dpt_array_df['filepath'].apply(os.path.basename)
    dpt_array_df_list.append(dpt_array_df)

In [ ]:
dataset = pd.concat(dpt_array_df_list)
dataset.head()

#### Convert the DPT array's numerical date to datetime field
The `date` field is stored as days since 0001-01-01 UTC (otherwise considered January 1, 1970). Here we change that to the common form of the date for better comprehension of the timescale.

In [ ]:
dataset['dt'] = pd.to_datetime(dates.num2date(dataset['date']), errors="coerce")
dataset.head()

### Check MERRA2 by plotting time series of temp & precip

The MERRA2 data we are using for this analysis consists of temperature and precipitation data. Plot the temporal extents of both datasets and see how precip and temperature vary over these extents.

In [ ]:
def plot_data(df, y_var, y_lab='Daily precip. [cm]', color='red'):
    df = df.dropna(subset=['dt'])
    x_var = 'dt'
    
    filenames = df['filename'].unique()
    n_files = len(filenames)
    ncols = min(n_files, 2)
    nrows = math.ceil(n_files / ncols)
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                             figsize=(10, 4 * nrows),
                             facecolor='white')
    axes = axes.flatten() if n_files > 1 else [axes]
    
    for i, fname in enumerate(filenames):
        ax = axes[i]
        subset = df[df['filename'] == fname]
        
        ax.scatter(subset[x_var], subset[y_var],
                   s=2, alpha=0.1, color=color)
                
        ax.set_ylim(0, 100)
        # ax.set_xlim(pd.Timestamp('2016-01-01'), pd.Timestamp('2025-12-31'))
        ax.set_xlabel(x_var)
        ax.set_ylabel(y_lab)
        ax.set_title(fname)
        
        # theme_bw equivalent
        ax.set_facecolor('white')
        ax.grid(True, color='lightgrey', linewidth=0.5)
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
    
    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    return fig

In [ ]:
fig = plot_data(dataset, 'precip', y_lab='Daily precip. [cm]', color='blue')
plt.show()

fig = plot_data(dataset, 'temp', y_lab='Daily temp. [C]', color='red')
plt.show()

## 7. Compare MERRA2 and HLS data
The purpose of using the MERRA2 data is to aid in the analysis of the HLS composites. By comparing trends in NDVI, EVI, and EVI2 to trends in real temperature and precipitation, we can determine the relationship between the variables and if outliers are caused by climatic inconsistencies.

In [ ]:
# Load MERRA2 -- chose this file because the spatial plotting earlier revealed it to be the one that HLS tile 39LUD sits in
# TODO: **** edit later ****
arr = np.load('/shared/users/sapolinsky/MERRA2_files/ignore_for_now/DPTarr_MERRA2_E0493_S0180.npy')
# arr = np.load('/shared/users/sapolinsky/MERRA2_files/DPTarr_MERRA2_E0500_S0154.npy')

# create a pandas dataframe of the date, precipitation, and temperature from the MERRA2 numpy file
merra2_df = pd.DataFrame({
    'date': pd.to_datetime(dates.num2date(arr[:, 0]), errors='coerce'),
    'precip': arr[:, 1],
    'temp': arr[:, 2]
}).set_index('date')

merra2_df.index = merra2_df.index.tz_localize(None)  # remove timezone to match HLS dates

# Aggregate MERRA2 to 10-day means/sums to match HLS cadence
hls_dates = pd.read_csv(clipped_roi_ndvi_file.with_suffix('').as_posix() + "_dates.csv", header=None, parse_dates=[0]).squeeze()
hls_dates = pd.to_datetime(hls_dates).sort_values().reset_index(drop=True)

records = []
for i, start in enumerate(hls_dates):
    # end is the day before the next start date (or +10 days for the last)
    end = hls_dates.iloc[i+1] if i+1 < len(hls_dates) else start + pd.Timedelta(days=10)
    window = merra2_df[(merra2_df.index >= start) & (merra2_df.index < end)]
    records.append({
        'date': start,
        'precip': window['precip'].mean(),
        'temp': window['temp'].mean()
    })

merra2_10day = pd.DataFrame(records).set_index('date')

# Aggregate HLS spatially (mean over all pixels per time step)
ndvi_ts = tenday_ndvis.mean(dim=['x', 'y'])
ndvi_df = ndvi_ts.to_dataframe(name='ndvi').reset_index().set_index('time')

evi_ts = tenday_evis.mean(dim=['x', 'y'])
evi_df = evi_ts.to_dataframe(name='evi').reset_index().set_index('time')

evi2_ts = tenday_evi2s.mean(dim=['x', 'y'])
evi2_df = evi2_ts.to_dataframe(name='evi2').reset_index().set_index('time')

# Merge on date
merged = merra2_10day.join(ndvi_df['ndvi'], how='inner')
merged = merged.join(evi_df['evi'], how='inner')
merged = merged.join(evi2_df['evi2'], how='inner')

#### Plot precipitation, temperature, NDVI, EVI, EVI2 data
These plots provide an overview of the data and to see how the vegetation indices change as climate changes.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 8), sharex=True)

axes[0].plot(merged.index, merged['precip'], color='blue')
axes[0].set_ylabel('Precip [cm]')

axes[1].plot(merged.index, merged['temp'], color='red')
axes[1].set_ylabel('Temp [°C]')

axes[2].plot(merged.index, merged['ndvi'], color='green')
axes[2].set_ylabel('NDVI')

axes[3].plot(merged.index, merged['evi'], color='orange')
axes[3].set_ylabel('EVI')

axes[4].plot(merged.index, merged['evi2'], color='purple')
axes[4].set_ylabel('EVI2')

for ax in axes:
    ax.grid(True, color='lightgrey', linewidth=0.5)
    ax.set_facecolor('white')

plt.xlabel('Date')
plt.tight_layout()
plt.show()

#### Vegetation Index vs. Precip/Temp
We want to see how/if EVI, NDVI, and EVI2 change with changing teperature and precipitation by plotting EVI, EVI2, and NDVI on the y-axis and the climate variables on the x-axes. 

In [ ]:
# Chart of how NDVI, EVI, and EVI2 change with changing precipitation and temperature
merged_clean = merged.dropna()
fig, ax = plt.subplots(2, 1)
ax[0].scatter(merged_clean['precip'], merged_clean['ndvi'], alpha=0.5)
ax[0].scatter(merged_clean['precip'], merged_clean['evi'], alpha=0.5)
ax[0].scatter(merged_clean['precip'], merged_clean['evi2'], alpha=0.5)
ax[0].set_ylabel('NDVI, EVI, EVI2')
ax[0].set_xlabel('Precipitation (cm)')
ax[0].legend(['NDVI', 'EVI', 'EVI2'], loc='lower right')

ax[1].scatter(merged_clean['temp'], merged_clean['ndvi'], alpha=0.5)
ax[1].scatter(merged_clean['temp'], merged_clean['evi'], alpha=0.5)
ax[1].scatter(merged_clean['temp'], merged_clean['evi2'], alpha=0.5)
ax[1].set_ylabel('NDVI, EVI, EVI2')
ax[1].set_xlabel('Temperature (C)')
ax[1].legend(['NDVI', 'EVI', 'EVI2'], loc='lower right')

plt.tight_layout()
plt.show()

As these plots show, for this site, there is very little (or no) correlation between vegetation indices and precipitation/temperature. This could be due to this site being vegetated year-round, or there being relatively little variation in temperature throughout the year (min = about 16˚C or 60˚F, max = about 25˚C or 77˚F). 

#### Monthly averages

In [ ]:
# monthly climate from daily MERRA2
climate_monthly = merra2_df.resample('ME').agg(
    {'precip': 'sum', 'temp': ['mean', 'min', 'max']})
climate_monthly.columns = ['precip_sum', 'temp_mean', 'temp_min', 'temp_max']

# monthly vegetation from 10-day HLS
veg_monthly = pd.DataFrame({
    'ndvi': ndvi_df['ndvi'].resample('ME').mean(),
    'evi':  evi_df['evi'].resample('ME').mean(),
    'evi2': evi2_df['evi2'].resample('ME').mean()
})

# merge at monthly level
monthly_all = climate_monthly.join(veg_monthly, how='inner')
monthly_all = monthly_all['2016':'2025']
monthly_all

#### Cloud cover
Cloud cover here is defined as the percentage of NaN (not a number) pixels in a composite. This may not be entirely representative of actual cloud data given that clouds, shadows, and other non-vegetated pixels were all masked out in the creation of the 10-day HLS composites. 

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# average across all pixels at each timestep, ignoring NaNs
ndvi_mean = tenday_ndvis.mean(dim=["x", "y"], skipna=True)
evi_mean = tenday_evis.mean(dim=["x", "y"], skipna=True)
evi2_mean = tenday_evi2s.mean(dim=["x", "y"], skipna=True)

# count NaN pixels at each timestep, express as % of total pixels
total_pixels = tenday_ndvis.sizes["x"] * tenday_ndvis.sizes["y"]
nan_count = tenday_ndvis.isnull().sum(dim=["x", "y"])
cloud_pct = (nan_count / total_pixels) * 100

# time axis
time = tenday_ndvis.time.values
time_evi = tenday_evis.time.values

# vegetation indices
# cloud cover as background shading 
# high cloud = tall shading from the top
axes[0].fill_between(time, cloud_pct/100, alpha=0.15, label="Cloud cover % (NaN pixels)", zorder=1)
axes[0].plot(time, ndvi_mean, label='NDVI')
axes[0].plot(time_evi, evi_mean, label='EVI')
axes[0].plot(time, evi2_mean, label='EVI2')
axes[0].set_ylabel('Vegetation Index')
axes[0].legend(loc='lower left')

# precipitation
axes[1].bar(monthly_all.index, monthly_all['precip_sum'], width=20)
axes[1].set_ylabel('Monthly Precip [cm]')

# temperature
axes[2].plot(monthly_all.index, monthly_all['temp_mean'], color='black', label='mean')
axes[2].plot(monthly_all.index, monthly_all['temp_min'], color='blue', label='min')
axes[2].plot(monthly_all.index, monthly_all['temp_max'], color='red', label='max')
axes[2].set_ylabel('Temp [°C]')
axes[2].legend()

for ax in axes:
    ax.grid(True, color='lightgrey', linewidth=0.5)
    ax.set_facecolor('white')

plt.xlabel('Date')
plt.tight_layout()
plt.show()

The plot above represents another way of exploring the data we have. By taking monthly aggregations of precipitation (total) and temperature (mean, min, max), and comparing that with how EVI, EVI2, and NDVI vary and how much cloud cover (NaN pixels) there is, these can reveal additional correlations between data.

#### Z-score plot
To get a better view of how temperature, precipitation, and NDVI/EVI/EVI2 vary from their average over time, here we calculate and plot the z-score of each.

**Z-score**: measures how many standard deviations a specific data point is above or below the mean

In [ ]:
# drop rows where any column is NaN for clean correlation
monthly_clean = monthly_all.dropna()

# Z-scores
ndvi_z = stats.zscore(monthly_clean['ndvi'])
precip_z = stats.zscore(monthly_clean['precip_sum'])
temp_z = stats.zscore(monthly_clean['temp_mean'])
evi_z = stats.zscore(monthly_clean['evi'])
evi2_z = stats.zscore(monthly_clean['evi2'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_clean.index, ndvi_z, label='NDVI', color='green')
ax.plot(monthly_clean.index, evi_z, label='EVI', color='darkgreen')
ax.plot(monthly_clean.index, evi2_z, label='EVI2', color='lightgreen')
ax.plot(monthly_clean.index, precip_z, label='Precip', color='steelblue')
ax.plot(monthly_clean.index, temp_z, label='Temp', color='red', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_ylabel('Z-score')
ax.legend()
ax.set_title('Normalized Climate vs Vegetation')
plt.tight_layout()
plt.show()

The z-score plot makes it clear that the dip in EVI, EVI2, and NDVI in 2022 was not caused by climatic variables. This indicates it was likely due to an error in the creation of the HLS 10-day composite. 

#### Lagged correlation
This is used to determine if there is a lagged relationship between climatic data and vegetation indices. 

In [ ]:
# lagged correlation: precip vs veg index at different lags (in months)
print("Lagged correlation (precip -> EVI):")
for lag in range(0, 10):
    corr = monthly_clean['evi'].corr(monthly_clean['precip_sum'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

print("Lagged correlation (precip -> EV2):")
for lag in range(0, 10):
    corr = monthly_clean['evi2'].corr(monthly_clean['precip_sum'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

Let's only look at the lagged correlation for anomoly values (subtract the monthly mean across all years from the value of that month to see how far from the mean it falls). This lets us see if in months with abnormally high or low precipitation (or temperature), if that is correlated with abnormally higher or lower vegetation indices.

In [ ]:
# compute monthly climatology and subtract
monthly_mean = monthly_clean.groupby(monthly_clean.index.month).mean()
anomalies = monthly_clean.copy()
for month in range(1, 13):
    mask = monthly_clean.index.month == month
    anomalies.loc[mask] = monthly_clean.loc[mask] - monthly_mean.loc[month]

# lagged correlation: precip vs veg index at different lags (in months)
print("Lagged correlation (precip -> EVI):")
for lag in range(0, 10):
    corr = anomalies['evi'].corr(anomalies['precip_sum'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

print("Lagged correlation (precip -> EV2):")
for lag in range(0, 10):
    corr = anomalies['evi2'].corr(anomalies['precip_sum'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

# lagged correlation: temp vs veg index at different lags (in months)
print("Lagged correlation (temp -> EVI):")
for lag in range(0, 10):
    corr = anomalies['evi'].corr(anomalies['temp_mean'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

print("Lagged correlation (temp -> EV2):")
for lag in range(0, 10):
    corr = anomalies['evi2'].corr(anomalies['temp_mean'].shift(lag))
    print(f"  lag {lag} months: r = {corr:.3f}")

## 8. Exploring different interpolation methods
As we have observed, there are many NaN pixel values in the HLS composite images. In this section we will explore different methods of interpolation to mask these NaN values.

**Interpolation**: estimating unknown values to create continuous surface representations

***Linear interpolation***:
- Given two data points, $(x_a, y_a)$ and $(x_b, y_b)$, interpolation of the two is given by: $y = y_a + (y_b - y_a) \frac{x - x_a}{x_b - x_a}$ at the point $(x, y)$
- Linear interpolation is fast and easy to compute, but not very precise.
- Connects data points with straight lines. 

***Spline interpolation***:
- Uses low-degree polynomials in each interval $[x_k,x_{k+1}]$ and ensures the curve is smooth.

***Polynomial interpolation***:
- Uses a single high-degree polynomial to fit all data points.


In [ ]:
# https://docs.xarray.dev/en/latest/generated/xarray.DataArray.interpolate_na.html
#  - linear calls numpy.interp

# linear, polynomial, spline
def interpolation(data, method_interp="spline"):
    if method_interp == "linear":
        # This method only appears to fill forward so some earlier-in-timeseries pxiels remain NaN. Each time step becomes "fuller" as NaNs have values
        #interpolated_evi = monthly_evis.interpolate_na(dim = 'time', method = 'linear')
        # bidirectional interpolation
        interpolated_vi = data.interpolate_na(
            dim = 'time', 
            method = 'linear', # linear is computationally lighter
            period = 36 # np.interp(method = 'linear') 
        )
    
    elif method_interp == "polynomial":
        # OPTION: polynomial
        interpolated_vi = data.interpolate_na(
            dim='time', 
            method='polynomial', 
            order=2,
            fill_value = 'extrapolate'
        )
        
    elif method_interp == "spline":
        # OPTION: cubic spline
        interpolated_vi = data.interpolate_na(
            dim='time', 
            method='cubic', 
            fill_value = 'extrapolate'
        )
        # clamp to valid veg index range after extrapolation
        interpolated_vi = interpolated_vi.clip(-1, 1)

    return interpolated_vi

In [ ]:
divnorm=colors.TwoSlopeNorm(vmin=-0.4, vcenter=0., vmax=1)

To shorten this notebook, use the variables `selected_index` and `selected_method` to select which combination of EVI/EVI2/NDVI and spline/linear/polynomial interpolations you want to look at. 

In [ ]:
# configuration: change these to run different combinations
methods = ["spline", "linear", "polynomial"]
veg_indices = {
    "ndvi": tenday_ndvis,
    "evi": tenday_evis,
    "evi2": tenday_evi2s
}
selected_index = "evi2"  # change this to switch index
const = 12 * 0 + 3       # timestep offset for spatial plots

raw_datacube = veg_indices[selected_index]

#### Plot specific pixels, original vs. interpolated

The following function is used to select random pixels to determine the differences between different interpolation methods and non-interpolated values. It is used for the 3 interpolation plots over time, and ranodm pixel time series graphs later in the notebook. 

In [ ]:
def sample_pixel_pairs(data, original_series, n=500):
    """
    Sample n random (x,y) coordinate pairs from actual pixel locations
    """
    np.random.seed(42)
    
    # Get all coordinate values
    x_coords = data.x.values
    y_coords = data.y.values
    
    # Create all possible (x,y) pairs - this represents actual pixels
    xx, yy = np.meshgrid(x_coords, y_coords)
    all_x_coords = xx.flatten()
    all_y_coords = yy.flatten()
    
    # Total number of pixels
    total_pixels = len(all_x_coords)
    print(f"Total pixels: {total_pixels} ({len(y_coords)} x {len(x_coords)})")
    
    # Randomly sample pixel indices
    random_pixel_indices = np.random.choice(total_pixels, size=n, replace=False)
    
    # Get the sampled coordinate pairs
    sampled_x = all_x_coords[random_pixel_indices]
    sampled_y = all_y_coords[random_pixel_indices]
    
    print(f"Sampling {n} actual pixel locations ({n/total_pixels*100:.2f}% of total)")
    
    # Extract time series for each sampled pixel
    sampled_data = []
    valid_count = 0
    
    for i in range(n):
        pixel_ts = data.sel(x=sampled_x[i], y=sampled_y[i], method='nearest')
        if not np.isnan(pixel_ts.values).all():
            sampled_data.append(pixel_ts.values)
            valid_count += 1
    
    print(f"Valid pixels: {valid_count} out of {n} sampled")
    
    # Calculate statistics
    sampled_array = np.array(sampled_data)
    median_ts = np.nanmedian(sampled_array, axis=0)
    std_ts = np.nanstd(sampled_array, axis=0)
    
    roi_med_vi = original_series.median(['x', 'y'], skipna=True)
    rmse = calculate_rmse(roi_med_vi, median_ts)
    
    return roi_med_vi, median_ts, std_ts, (sampled_x, sampled_y), valid_count, rmse

#### Interpolated samples vs. original median

This chart shows how the different interpolation methods vary from the median ROI vegetation index of choosing. 
The **root mean squared error (RMSE)** tells you how far away interpolated predictions fall from measured true values.

In [ ]:
def calculate_rmse(observed, predicted):
    """
    Calculate RMSE (Root Mean Square Error) between two time series
    """
    rmse = np.sqrt(np.nanmean((observed - predicted)**2))
    return rmse

In [ ]:
# run interpolation for all methods on selected index
interpolated = {}
stats = {}
for method in methods:
    interp = interpolation(raw_datacube, method)
    roi_med, median, std, coords, val, rmse = sample_pixel_pairs(interp, raw_datacube, n=10000)
    interpolated[method] = interp
    stats[method] = {
        "roi_med": roi_med,
        "median": median,
        "std": std,
        "coords": coords,
        "val": val,
        "rmse": rmse
    }

In [ ]:
# interpolation comparison: median time series
fig, axes = plt.subplots(len(methods), 1, figsize=(10, 4 * len(methods)))
for ax, method in zip(axes, methods):
    s = stats[method]
    ax.axhline(y=0, color="black", alpha=0.7, linestyle="--")
    ax.plot(stats[method]["roi_med"].time, s["roi_med"], 
            label=f"Median ROI {selected_index.upper()}", color="red")
    ax.plot(interpolated[method].time, s["median"], 
            color="blue", label=f"Interpolated {method.capitalize()} (n={s['val']})")
    ax.fill_between(interpolated[method].time,
                    s["median"] - s["std"], s["median"] + s["std"],
                    alpha=0.3, color="blue", label="±1 Std Dev")
    ax.set_title(f"{method.capitalize()}, RMSE: {s['rmse']:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.xlabel("Time")
plt.suptitle(f"Interpolated {selected_index.upper()} Sample vs Original Median")
plt.tight_layout()
plt.show()

#### Plot ROI vegetation index for both interpolated and not

Using the `selected_method` of interpolation, plot the chosen vegetation index with its interpolated form and non-interpolated form to see how they vary. 

In [ ]:
# interpolated vs. original
# one selected method for spatial comparison — change as needed
selected_method = "linear"

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, ax in enumerate(axes.flatten()):
    interpolated[selected_method].isel(time=i + const).plot(
        ax=ax, norm=divnorm, cmap="RdYlGn")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.suptitle(f"Interpolated {selected_index.upper()}: {selected_method.capitalize()}", size=16)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, ax in enumerate(axes.flatten()):
    raw_datacube.isel(time=i + const).plot(ax=ax, norm=divnorm, cmap="RdYlGn")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.suptitle(f"10-Day {selected_index.upper()} Composites", size=16)
plt.tight_layout()
plt.show()

As you can see, some 10-day composites are very sparse, requring the interpolation to have to fill in a lot of the missing pixels. 

#### 

In [ ]:
# difference maps
raw_filled = raw_datacube.fillna(0)
diffs = {method: (raw_filled - interpolated[method]) for method in methods}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, ax in enumerate(axes.flatten()):
    diffs[selected_method].isel(time=i + const).plot(
        ax=ax, vmin=-0.1, vmax=0.1, center=0, cmap="RdBu_r")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.suptitle(f"Interpolated {selected_index.upper()} Diff: Original - Interpolated "
             f"(+: higher fill / -: lower fill)", size=16)
plt.tight_layout()
plt.show()

#### Plot specific pixels, original vs. interpolated

Plot randomly-selected pixels to see how each interpolation method varies in its values. 

In [ ]:
# random pixel timeseries
np.random.seed(8)
n = 10

# get valid pixel indices from first timestep of selected method
valid_mask = interpolated[selected_method].isel(time=0).notnull()
valid_y_idx, valid_x_idx = np.where(valid_mask.values)
random_indices = np.random.choice(len(valid_x_idx), size=n, replace=False)

fig, axes = plt.subplots(5, 2, figsize=(15, 15))
for i, ax in enumerate(axes.flatten()):
    ax.axhline(y=0, color="black", alpha=0.7, linestyle="--")
    ax.plot(raw_datacube.time, 
            raw_datacube.mean(["x", "y"], skipna=True),
            label=selected_index.upper(), color="grey", alpha=0.8)
    for method in methods:
        pixel_ts = interpolated[method].isel(
            x=valid_x_idx[random_indices[i]],
            y=valid_y_idx[random_indices[i]])
        ax.plot(pixel_ts.time, pixel_ts.values, label=method)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle(f"Random Pixel Time Series: {selected_index.upper()}")
plt.tight_layout()
plt.show()

## 9. Annual Phenometrics

**Phenometrics**: phenological (cyclic and natural phenomena) transition dates

Phenometrics track seasonal plant cycles by analyzing vegetation indices. In this analysis, we will look at:
- How the maximum vegetation index value for each pixel in the ROI varies over the time frame of the composites.
- What day of the year each pixel experiences its maximum across the years.

These provide additional insights into the how the data varies spatially and over time. 

In [ ]:
def adjust_to_10d_midpoint(data):
    """
    Adjust timestamps from 1st of 10-day period to midpoint of 10-day period
    """
    original_times = pd.to_datetime(data.time.values)
    midpoint_times = []
    
    # return data_adjusted
    for i, time in enumerate(original_times):
        # determine end of this composite window
        # end is either day before next composite's start, or Dec 31 if last of year
        if i + 1 < len(original_times):
            next_time = original_times[i + 1]
            if next_time.year == time.year:
                # next composite is same year — window ends day before it starts
                window_end = next_time - pd.Timedelta(days=1)
            else:
                # last composite of the year
                window_end = pd.Timestamp(year=time.year, month=12, day=31)
        else:
            # very last composite in the entire dataset
            window_end = pd.Timestamp(year=time.year, month=12, day=31)

        window_length = (window_end - time).days + 1
        midpoint = time + pd.Timedelta(days=window_length // 2)
        midpoint_times.append(midpoint)

    return data.assign_coords(time=midpoint_times)

def find_annual_max_doy_with_fill(data, fill_value=-9999):
    """
    Fill NaNs with a low value, then find annual max DOY
    """
    data_cadence_midpoint = adjust_to_10d_midpoint(data)
    
    # Fill NaNs with very low value
    data_filled = data_cadence_midpoint.fillna(fill_value)
    
    # Now groupby operations work smoothly
    annual_max_doy_list = []
    years = []
    
    for year, year_data in data_filled.groupby('time.year'):
        # Find argmax (will skip -9999 filled pixels naturally)
        max_indices = year_data.argmax(dim='time')
        max_times = year_data.time[max_indices]
        max_doy = max_times.dt.dayofyear
        
        # mask out the -9999 pixels again
        original_nan_mask = data_cadence_midpoint.sel(time=year_data.time).isnull().all(dim='time')
        max_doy = max_doy.where(~original_nan_mask, np.nan)
        
        annual_max_doy_list.append(max_doy)
        years.append(year)
    
    result = xr.concat(annual_max_doy_list, dim='year')
    result = result.assign_coords(year=years)
    
    return result

In [ ]:
st_year = '2016-01-01'

def calculate_annual_phenometrics(datacube):

    annual_groupby = datacube.groupby('time.year')

    # Phenometrics
    annual_mean = annual_groupby.mean()
    annual_max = annual_groupby.max()
    annual_max_doy = find_annual_max_doy_with_fill(datacube)  
    
    phenometrics_ds = xr.Dataset({
        'Mean_VI': annual_mean,
        'Max_VI': annual_max,
        'Max_DOY_VI': annual_max_doy,
    })

    return phenometrics_ds

### Max EVI/EVI2/NDVI value per pixel & Max EVI/EVI2/NDVI DOY

The first set of tiles show how the maximum EVI, EVI2, or NDVI value (depending on what you select as the `selected_index`) for each pixel in the ROI varies over the time frame of the composites.

The second set of tiles show what day of the year (DOY) each pixel experiences its maximum EVI, EVI2, or NDVI value (depending on what you select as the `selected_index`) across the years.

In [ ]:
# phenometrics
lower = interpolated[selected_method].quantile(0.01)
upper = interpolated[selected_method].quantile(0.99)
interp_cleaned = interpolated[selected_method].clip(min=lower, max=upper)
phenometrics = calculate_annual_phenometrics(interp_cleaned.sel(time=slice(st_year, None)))

for metric, title in [("Max_VI", "Max Value"), ("Max_DOY_VI", "Max DOY")]:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    for i, ax in enumerate(axes.flatten()):
        phenometrics[metric][i].plot(ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel("")
    plt.suptitle(f"{title} - {selected_index.upper()} ({selected_method.capitalize()})", size=16)
    plt.tight_layout()
    plt.show()

Many of the pixels in the above plots seem to show the maximum EVI dates as near the end/beginning of each year. This makes sense because Madagascar is in the Southern Hemisphere, experiencing its warmest months from November-March. 

## 10. Takeaways

You now have a complete workflow to:
- Calculate EVI, EVI2, and NDVI for Harmonized Landsat-Sentinel (HLS) 10-day composite images.
- Visualize how EVI, EVI2, and NDVI change across the spatial and temporal extent.
- Import MERRA2 data and compare with HLS data.
  - Includes how to use MERRA2 data to inform analysis of HLS data.
- Methods to interpolate HLS data (spline, polynomial, linear).